In [3]:
# Alpha Connectome, sweep 9: a stronger mask-weighted teacher, born-again distillation, auxiliary targets and
# per-sequence normalised inputs. One A100 run (~3.5 h), self-contained (the whole pipeline is embedded below).
#   Phase A (5 passes, no teacher): 6 new teacher members + 4 test models (aux targets 0.3 / 1.0, norm inputs,
#            norm + aux), each comparable with t9_h128_s0.                                 -> runs/sweep9_teachers
#   Phase B: teacher t9 = greedy ensemble (validation half A) over every mask-weighted plain model so far
#            (sweeps 5-9, incl. minGRU / TCN); born-again teacher ba = average of the 6 existing students.
#   Phase C (10 passes): students of t9 (2 seeds, + aux, + norm) and of ba.                 -> runs/sweep9
# Re-runnable: finished models and cached teachers (runs/cache/t_9.npy, t_ba.npy) are skipped.
# Keep this tab open while it runs.
import os, subprocess, sys, json, glob, shutil, time
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/wunder-connectome/runs'
os.makedirs(OUT_DIR, exist_ok=True)

def drive_writable():
    """Drive can turn read-only (storage full, or the mount drops during the 30-min download): check before writing."""
    for attempt in range(2):
        try:
            with open(f'{OUT_DIR}/.write_test', 'w') as f: f.write('ok')
            os.remove(f'{OUT_DIR}/.write_test'); print('Drive is writable', flush=True); return
        except OSError as e:
            print(f'Drive not writable ({e}), remounting', flush=True)
            try: drive.flush_and_unmount()
            except Exception: pass
            drive.mount('/content/drive', force_remount=True)
    raise RuntimeError('Google Drive is read-only: most likely its storage is full. Free space (and empty the Drive '
                       'Trash) at drive.google.com/settings/storage, then re-run this cell; the dataset stays on this machine.')

drive_writable()

def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    assert p.wait() == 0, 'command failed: ' + cmd[:80]

sh('nvidia-smi --query-gpu=name,memory.total --format=csv; echo "CPUs: $(nproc)"; free -g | head -2')
DATA = r"""mkdir -p /content/wnn && cd /content/wnn
check() {
python3 - <<'EOF'
import sys, pyarrow.parquet as pq
ok = True
for f, n in {'train': 10607, 'valid': 1873, 'valid_mask': 1873}.items():
    try:
        g = pq.ParquetFile(f'datasets/{f}.parquet').metadata.num_row_groups
        print(f'{f}: {g} sequences', 'OK' if g == n else f'(expected {n})'); ok &= g == n
    except Exception as e:
        print(f'{f}: BROKEN ({type(e).__name__}: {e})'); ok = False
sys.exit(0 if ok else 1)
EOF
}
if check; then echo 'data OK, nothing to do'; else
  rm -rf datasets
  URL=https://files.wundernn.io/wnn_connectome_starterpack.tar.gz
  for i in $(seq 10); do   # -C - resumes where a dropped connection stopped
    curl -L -C - --retry 20 --retry-all-errors -sS -o pack.tar.gz "$URL" && break
    echo "download interrupted, resuming ($i)"; sleep 5
  done
  ls -la pack.tar.gz
  tar -xzf pack.tar.gz --strip-components=1 --warning=no-unknown-keyword && find . -name '._*' -delete
  if check; then rm pack.tar.gz; echo 'data OK'; else echo 'DATA STILL BROKEN: delete /content/wnn/pack.tar.gz and re-run this cell'; exit 1; fi
fi
du -sh datasets"""
sh(DATA)
sh('pip -q install onnx onnxruntime')
os.chdir('/content/wnn')

# the pipeline code (pipeline/*.py from the Mac, 2026-11-03)
FILES = {'pipeline/__init__.py': '',
 'pipeline/data.py': '"""Fast sequence loading from the competition parquet files.\n'
                     '\n'
                     'Each row group is one 20k-step sequence. Readers keep one ParquetFile per thread\n'
                     '(re-opening parses a large footer) and write straight into a preallocated batch array.\n'
                     '"""\n'
                     'import threading\n'
                     'from concurrent.futures import ThreadPoolExecutor\n'
                     '\n'
                     'import numpy as np\n'
                     'import pyarrow.parquet as pq\n'
                     '\n'
                     'from utils import FEATURE_COLUMNS, SEQUENCE_LENGTH, TARGET_COLUMNS, WARMUP\n'
                     '\n'
                     'N_FEATURES = len(FEATURE_COLUMNS)\n'
                     '\n'
                     '\n'
                     'class SeqStore:\n'
                     '    def __init__(self, path: str, n_threads: int = 4):\n'
                     '        self.path = path\n'
                     '        pf = pq.ParquetFile(path)\n'
                     '        self.n_seqs = pf.num_row_groups\n'
                     '        self.has_mask = "is_scored" in pf.schema_arrow.names\n'
                     '        self.cols = list(FEATURE_COLUMNS) + list(TARGET_COLUMNS) + (["is_scored"] if '
                     'self.has_mask else [])\n'
                     '        self._local = threading.local()\n'
                     '        self._pool = ThreadPoolExecutor(n_threads)\n'
                     '        self._cache = None\n'
                     '\n'
                     '    def enable_cache(self):\n'
                     '        """Keep every sequence in RAM after its first read (~9 MB each)."""\n'
                     '        self._cache = {}\n'
                     '\n'
                     '    def _pf(self):\n'
                     '        if not hasattr(self._local, "pf"):\n'
                     '            self._local.pf = pq.ParquetFile(self.path)\n'
                     '        return self._local.pf\n'
                     '\n'
                     '    def _load_into(self, i, g, X, Y, M):\n'
                     '        if self._cache is not None and g in self._cache:\n'
                     '            X[i], Y[i], M[i] = self._cache[g]\n'
                     '            return\n'
                     '        df = self._pf().read_row_group(int(g), columns=self.cols, '
                     'use_threads=False).to_pandas()\n'
                     '        a = df.to_numpy(np.float32)\n'
                     '        X[i] = a[:, :N_FEATURES]\n'
                     '        Y[i] = a[:, N_FEATURES:N_FEATURES + 2]\n'
                     '        need = np.arange(SEQUENCE_LENGTH) >= WARMUP\n'
                     '        M[i] = need & (a[:, -1] > 0.5) if self.has_mask else need\n'
                     '        if self._cache is not None:\n'
                     '            self._cache[g] = (X[i].copy(), Y[i].copy(), M[i].copy())\n'
                     '\n'
                     '    def load(self, ids):\n'
                     '        """-> X [B,T,112] f32, Y [B,T,2] f32, M [B,T] bool (scored rows; all required rows if no '
                     'mask)."""\n'
                     '        B = len(ids)\n'
                     '        X = np.empty((B, SEQUENCE_LENGTH, N_FEATURES), np.float32)\n'
                     '        Y = np.empty((B, SEQUENCE_LENGTH, 2), np.float32)\n'
                     '        M = np.empty((B, SEQUENCE_LENGTH), bool)\n'
                     '        list(self._pool.map(lambda ig: self._load_into(ig[0], ig[1], X, Y, M), enumerate(ids)))\n'
                     '        return X, Y, M\n'
                     '\n'
                     '    def batches(self, ids, batch_size):\n'
                     '        """Yield loaded batches, prefetching the next one in the background."""\n'
                     '        chunks = [ids[i:i + batch_size] for i in range(0, len(ids), batch_size)]\n'
                     '        if not chunks:\n'
                     '            return\n'
                     '        prefetch = ThreadPoolExecutor(1)\n'
                     '        fut = prefetch.submit(self.load, chunks[0])\n'
                     '        for k in range(len(chunks)):\n'
                     '            hold = [fut.result()]\n'
                     '            fut = prefetch.submit(self.load, chunks[k + 1]) if k + 1 < len(chunks) else None\n'
                     '            yield hold.pop()  # the generator keeps no reference, so the consumer can free the '
                     'batch\n'
                     '        prefetch.shutdown()\n'
                     '\n'
                     '\n'
                     'def split_train(n_seqs: int, n_holdout: int, seed: int = 0):\n'
                     '    """Deterministic split of train row groups into (train_ids, holdout_ids)."""\n'
                     '    perm = np.random.default_rng(seed).permutation(n_seqs)\n'
                     '    return np.sort(perm[n_holdout:]), np.sort(perm[:n_holdout])\n',
 'pipeline/ensemble.py': '"""Score models alone and averaged, pick an ensemble on the train holdout, optionally export '
                         'it.\n'
                         '\n'
                         '    python -m pipeline.ensemble --ckpts "runs/*/best.pt" --budgets 1 1.5 2 --export '
                         'runs/ensembles\n'
                         '\n'
                         'Every model is run once over the holdout and validation data (shared loading); combinations '
                         'are then\n'
                         'scored from the stored predictions. Members are rescaled to a common prediction spread '
                         'before\n'
                         'averaging (scales measured on the holdout). Selection uses the holdout only, so the '
                         'validation\n'
                         'numbers stay an honest check of the chosen combination.\n'
                         '\n'
                         'Cost = multiply-adds per row relative to the baseline (models.macs). Real run time also has '
                         'a\n'
                         "fixed per-row overhead, so an ensemble's measured time ratio is lower than its cost; check\n"
                         'candidates with pipeline.speed. One greedy ensemble is built and exported per --budgets '
                         'value.\n'
                         '"""\n'
                         'import argparse\n'
                         'import glob\n'
                         'import json\n'
                         'import os\n'
                         'import shutil\n'
                         'import time\n'
                         '\n'
                         'import numpy as np\n'
                         '\n'
                         'from utils import SEQUENCE_LENGTH, WARMUP, weighted_pearson\n'
                         '\n'
                         'from .data import SeqStore, split_train\n'
                         'from .fast_eval import load_ckpt, net_inputs, pick_device, predict_batch\n'
                         'from .model import BASELINE_MACS, EnsembleNet, export_onnx, feat_kind, load_baseline, macs\n'
                         '\n'
                         'HERE = os.path.dirname(os.path.abspath(__file__))\n'
                         'NEED = np.arange(SEQUENCE_LENGTH) >= WARMUP\n'
                         '\n'
                         '\n'
                         'def load_member(path):\n'
                         '    if path == "baseline":\n'
                         '        return "baseline", load_baseline("baseline/baseline.onnx")\n'
                         '    return os.path.basename(os.path.dirname(path)), load_ckpt(path)\n'
                         '\n'
                         '\n'
                         'def collect(nets, store, ids, device, masks):\n'
                         '    """-> Y {mask: [N,2]}, P {mask: [K,N,2]} for masks \'all\' (required rows) and/or '
                         '\'scored\'."""\n'
                         '    ys = {m: [] for m in masks}\n'
                         '    ps = {m: [[] for _ in nets] for m in masks}\n'
                         '    for X, Y, M in store.batches(list(ids), 128):\n'
                         '        cache = {}  # extras once per batch and kind\n'
                         '        sel = {"all": np.broadcast_to(NEED, M.shape), "scored": M}\n'
                         '        for m in masks:\n'
                         '            ys[m].append(Y[sel[m]])\n'
                         '        for k, net in enumerate(nets):\n'
                         '            P = predict_batch(net, net_inputs(net, X, cache), device)\n'
                         '            for m in masks:\n'
                         '                ps[m][k].append(P[sel[m]])\n'
                         '    return ({m: np.concatenate(ys[m]) for m in masks},\n'
                         '            {m: np.stack([np.concatenate(p) for p in ps[m]]) for m in masks})\n'
                         '\n'
                         '\n'
                         'def wp(y, p):\n'
                         '    return float(np.mean([weighted_pearson(y[:, j], p[:, j]) for j in range(2)]))\n'
                         '\n'
                         '\n'
                         'def combine(P, scales, idx):\n'
                         '    return (P[idx] * scales[idx][:, None, :]).mean(0)\n'
                         '\n'
                         '\n'
                         'def main():\n'
                         '    ap = argparse.ArgumentParser()\n'
                         '    ap.add_argument("--ckpts", nargs="+", required=True, help="best.pt paths/globs, or '
                         '\'baseline\'")\n'
                         '    ap.add_argument("--holdout", type=int, default=500)\n'
                         '    ap.add_argument("--valid-seqs", type=int, default=None, help="first N validation seqs '
                         '(default all)")\n'
                         '    ap.add_argument("--max-members", type=int, default=5)\n'
                         '    ap.add_argument("--budgets", type=float, nargs="+", default=[99.0],\n'
                         '                    help="max total cost (x baseline multiply-adds); one greedy ensemble per '
                         'budget")\n'
                         '    ap.add_argument("--export", help="root folder: writes <root>/cost<budget>/submission per '
                         'budget")\n'
                         '    ap.add_argument("--device", default="auto")\n'
                         '    ap.add_argument("--threads", type=int, default=4)\n'
                         '    ap.add_argument("--valid-split", help="split.json from mask_model: report on validation '
                         'half B")\n'
                         '    ap.add_argument("--select-on", choices=["holdout", "validA"], default="holdout",\n'
                         '                    help="what greedy selection maximises: holdout all-rows WP, or '
                         'validation half A scored rows "\n'
                         '                         "(needs --valid-split; right for mask-weighted models, keeps half B '
                         'honest)")\n'
                         '    args = ap.parse_args()\n'
                         '    assert args.select_on == "holdout" or args.valid_split, "--select-on validA needs '
                         '--valid-split"\n'
                         '\n'
                         '    paths = []\n'
                         '    for c in args.ckpts:\n'
                         '        paths += [c] if c == "baseline" else sorted(glob.glob(c))\n'
                         '    names, nets = zip(*[load_member(p) for p in paths])\n'
                         '    device = pick_device(args.device)\n'
                         '    nets = [n.eval().to(device) for n in nets]\n'
                         '    print(f"{len(nets)} models: {\', \'.join(names)}", flush=True)\n'
                         '\n'
                         '    t = time.time()\n'
                         '    train = SeqStore("datasets/train.parquet", args.threads)\n'
                         '    valid = SeqStore("datasets/valid.parquet", args.threads)\n'
                         '    _, hold_ids = split_train(train.n_seqs, args.holdout)\n'
                         '    Yh, Ph = collect(nets, train, hold_ids, device, ["all"])\n'
                         '    if args.valid_split:\n'
                         '        split = json.load(open(args.valid_split))\n'
                         '        Ya, Pa = collect(nets, valid, split["A"], device, ["scored"])\n'
                         '        Yv, Pv = collect(nets, valid, split["B"], device, ["scored", "all"])  # report on '
                         'half B\n'
                         '    else:\n'
                         '        Yv, Pv = collect(nets, valid, range(args.valid_seqs or valid.n_seqs), device, '
                         '["scored", "all"])\n'
                         '    print(f"predictions collected in {time.time() - t:.0f}s", flush=True)\n'
                         '    SEL = args.select_on\n'
                         '\n'
                         '    sd = Ph["all"].std(axis=1)  # [K, 2]\n'
                         '    scales = sd.mean(0, keepdims=True) / sd\n'
                         '\n'
                         '    def sel_score(idx):\n'
                         '        if SEL == "validA":\n'
                         '            return wp(Ya["scored"], combine(Pa["scored"], scales, idx))\n'
                         '        return wp(Yh["all"], combine(Ph["all"], scales, idx))\n'
                         '\n'
                         '    def score(idx):  # valid_* = half B when --valid-split is given\n'
                         '        out = {"holdout": wp(Yh["all"], combine(Ph["all"], scales, idx)),\n'
                         '               "valid_scored": wp(Yv["scored"], combine(Pv["scored"], scales, idx)),\n'
                         '               "valid_all": wp(Yv["all"], combine(Pv["all"], scales, idx))}\n'
                         '        if args.valid_split:\n'
                         '            out["validA"] = wp(Ya["scored"], combine(Pa["scored"], scales, idx))\n'
                         '        return out\n'
                         '\n'
                         '    cost = np.array([macs(n.config) / BASELINE_MACS for n in nets])\n'
                         '    rows = []\n'
                         '    singles = [(k, score([k])) for k in range(len(nets))]\n'
                         '    for k, sc in singles:\n'
                         '        rows.append({"combo": names[k], "n": 1, "cost": cost[k], **sc})\n'
                         '    order = [k for k, sc in sorted(singles, key=lambda ks: -ks[1][SEL])]\n'
                         '    for top in range(2, min(len(nets), args.max_members) + 1):\n'
                         '        rows.append({"combo": f"top{top} by {SEL}", "n": top, "cost": '
                         'cost[order[:top]].sum(),\n'
                         '                     "members": [names[k] for k in order[:top]], **score(order[:top])})\n'
                         '\n'
                         '    def greedy(budget):\n'
                         '        """Forward selection on SEL (no repeats) within a total cost budget."""\n'
                         '        ok = [k for k in order if cost[k] <= budget]\n'
                         '        if not ok:\n'
                         '            return []\n'
                         '        chosen, best = [ok[0]], singles[ok[0]][1][SEL]\n'
                         '        while len(chosen) < args.max_members:\n'
                         '            cand = [(k, sel_score(chosen + [k])) for k in range(len(nets))\n'
                         '                    if k not in chosen and cost[chosen + [k]].sum() <= budget]\n'
                         '            if not cand:\n'
                         '                break\n'
                         '            k, sc = max(cand, key=lambda c: c[1])\n'
                         '            if sc <= best:\n'
                         '                break\n'
                         '            chosen.append(k)\n'
                         '            best = sc\n'
                         '        return chosen\n'
                         '\n'
                         '    picks = {}\n'
                         '    for b in args.budgets:\n'
                         '        chosen = greedy(b)\n'
                         '        if chosen:\n'
                         '            picks[b] = chosen\n'
                         '            rows.append({"combo": f"greedy, cost <= {b:g}", "n": len(chosen), "cost": '
                         'cost[chosen].sum(),\n'
                         '                         "members": [names[k] for k in chosen], **score(chosen)})\n'
                         '\n'
                         '    vl = "B" if args.valid_split else ""\n'
                         '    # diversity: members that disagree with the best single model are the ones an average '
                         '(or a teacher) gains from\n'
                         '    best_k = order[0]\n'
                         '    print(f"\\nprediction correlation with {names[best_k]} (valid{vl} scored rows, mean of '
                         't0/t1):")\n'
                         '    for k in order:\n'
                         '        r = np.mean([np.corrcoef(Pv["scored"][k][:, j], Pv["scored"][best_k][:, j])[0, 1] '
                         'for j in range(2)])\n'
                         '        print(f"  {names[k][:40]:40s} {r:.4f}")\n'
                         '    print(f"\\n{\'combo\':30s} {\'n\':>2s} {\'cost\':>5s} {\'holdout\':>8s} '
                         '{\'valid\'+vl+\'_sc\':>9s} {\'valid\'+vl+\'_all\':>10s}"\n'
                         '          + (f" {\'validA_sc\':>9s}" if args.valid_split else ""))\n'
                         '    for r in rows:\n'
                         '        print(f"{r[\'combo\'][:30]:30s} {r[\'n\']:2d} {r[\'cost\']:5.2f} '
                         '{r[\'holdout\']:8.5f} {r[\'valid_scored\']:9.5f} "\n'
                         '              f"{r[\'valid_all\']:10.5f}" + (f" {r[\'validA\']:9.5f}" if "validA" in r else '
                         '"")\n'
                         '              + (f"   {\'+\'.join(r[\'members\'])}" if "members" in r else ""))\n'
                         '\n'
                         '    if args.export:\n'
                         '        for b, chosen in picks.items():\n'
                         '            out = os.path.join(args.export, f"cost{b:g}", "submission")\n'
                         '            os.makedirs(out, exist_ok=True)\n'
                         '            net = nets[chosen[0]] if len(chosen) == 1 else EnsembleNet([nets[k] for k in '
                         'chosen], scales[chosen])\n'
                         '            if len({feat_kind(nets[k]) for k in chosen} - {None}) <= 1:\n'
                         '                export_onnx(net, os.path.join(out, "model.onnx"))\n'
                         '                shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(out, '
                         '"solution.py"))\n'
                         '                shutil.copy(os.path.join(HERE, "features.py"), os.path.join(out, '
                         '"wnn_features.py"))\n'
                         '            else:\n'
                         '                print(f"cost<={b:g}: members use different extra features, no ONNX '
                         '(ensemble.json only, e.g. for a teacher)")\n'
                         '            row = next(r for r in rows if r["combo"] == f"greedy, cost <= {b:g}")\n'
                         '            json.dump({"members": [paths[k] for k in chosen], "scales": '
                         'scales[chosen].tolist(),\n'
                         '                       "scores": {k: (float(v) if isinstance(v, (float, np.floating)) else '
                         'v) for k, v in row.items()}},\n'
                         '                      open(os.path.join(out, "ensemble.json"), "w"), indent=1)\n'
                         '            print(f"exported cost<={b:g}: {len(chosen)} model(s) -> {out}")\n'
                         '\n'
                         'if __name__ == "__main__":\n'
                         '    main()\n',
 'pipeline/fast_eval.py': '"""Batched evaluation: runs whole sequences through the network at once (seconds instead '
                          'of\n'
                          'the ~15 min row-by-row loop) and scores them with the official GlobalAccumulator.\n'
                          '\n'
                          '    python -m pipeline.fast_eval --baseline                 # should reproduce 0.617052\n'
                          '    python -m pipeline.fast_eval --ckpt runs/NAME/best.pt --holdout 500\n'
                          '"""\n'
                          'import argparse\n'
                          'import json\n'
                          'import time\n'
                          '\n'
                          'import numpy as np\n'
                          'import torch\n'
                          '\n'
                          'from utils import SEQUENCE_LENGTH, WARMUP, GlobalAccumulator\n'
                          '\n'
                          'from .data import SeqStore, split_train\n'
                          'from .model import build_net, feat_kind, load_baseline\n'
                          '\n'
                          'NEED = np.arange(SEQUENCE_LENGTH) >= WARMUP\n'
                          '\n'
                          '\n'
                          'def pick_device(name="auto"):\n'
                          '    if name != "auto":\n'
                          '        return torch.device(name)\n'
                          '    if torch.cuda.is_available():\n'
                          '        return torch.device("cuda")\n'
                          '    if torch.backends.mps.is_available():\n'
                          '        return torch.device("mps")\n'
                          '    return torch.device("cpu")\n'
                          '\n'
                          '\n'
                          'def with_extras(X, kind="act"):\n'
                          '    """Append the extra causal features of `kind` (pipeline/features.py) to raw inputs '
                          '[B,T,112]."""\n'
                          '    from .features import offline\n'
                          '    return np.concatenate([X, offline(X[..., :112], kind)], -1)\n'
                          '\n'
                          '\n'
                          'def net_inputs(net, X, cache):\n'
                          '    """Raw batch X [B,T,112] -> what `net` takes; `cache` (a dict per batch) shares the '
                          'extras between models."""\n'
                          '    kind = feat_kind(net)\n'
                          '    if kind and kind not in cache:\n'
                          '        cache[kind] = with_extras(X, kind)\n'
                          '    return cache[kind] if kind else X\n'
                          '\n'
                          '\n'
                          '@torch.no_grad()\n'
                          'def predict_batch(net, X, device, chunk=2000):\n'
                          '    if X.shape[-1] == 112:\n'
                          '        X = net_inputs(net, X, {})\n'
                          '    B, T, _ = X.shape\n'
                          '    out = np.empty((B, T, 2), np.float32)\n'
                          '    h = xp = None\n'
                          '    for s in range(0, T, chunk):\n'
                          '        x = torch.from_numpy(X[:, s:s + chunk]).to(device)\n'
                          '        p, h = net(x, h, xp)\n'
                          '        xp = x[:, -1]\n'
                          '        out[:, s:s + chunk] = p.float().cpu().numpy()\n'
                          '    return out\n'
                          '\n'
                          '\n'
                          'def evaluate_many(nets, store: SeqStore, ids, device, batch_size=128):\n'
                          '    """Score several models in one pass over the data. Per model:\n'
                          "    {'scored': WP on scored rows, 'all': WP on all required rows, ...} (equal without a "
                          'mask)."""\n'
                          '    for net in nets:\n'
                          '        net.eval()\n'
                          '    accs = [(GlobalAccumulator(), GlobalAccumulator()) for _ in nets]\n'
                          '    for X, Y, M in store.batches(list(ids), batch_size):\n'
                          '        cache = {}  # extras once per batch and kind, shared by every model that uses them\n'
                          '        for net, (acc, acc_all) in zip(nets, accs):\n'
                          '            P = predict_batch(net, net_inputs(net, X, cache), device)\n'
                          '            for b in range(len(X)):\n'
                          '                acc.add(Y[b], P[b], M[b])\n'
                          '                if store.has_mask:\n'
                          '                    acc_all.add(Y[b], P[b], NEED)\n'
                          '    out = []\n'
                          '    for acc, acc_all in accs:\n'
                          '        r = acc.result()\n'
                          '        res = {"scored": r["weighted_pearson"], "t0": r["t0"], "t1": r["t1"], "n_seqs": '
                          'r["blocks"]}\n'
                          '        res["all"] = acc_all.result()["weighted_pearson"] if store.has_mask else '
                          'res["scored"]\n'
                          '        out.append(res)\n'
                          '    return out\n'
                          '\n'
                          '\n'
                          'def evaluate(net, store: SeqStore, ids, device, batch_size=128):\n'
                          '    return evaluate_many([net], store, ids, device, batch_size)[0]\n'
                          '\n'
                          '\n'
                          'def load_ckpt(path):\n'
                          '    ck = torch.load(path, map_location="cpu", weights_only=False)\n'
                          '    net = build_net(ck["config"])\n'
                          '    net.load_state_dict(ck["state_dict"])\n'
                          '    return net\n'
                          '\n'
                          '\n'
                          'def main():\n'
                          '    ap = argparse.ArgumentParser()\n'
                          '    g = ap.add_mutually_exclusive_group(required=True)\n'
                          '    g.add_argument("--baseline", action="store_true")\n'
                          '    g.add_argument("--ckpt")\n'
                          '    ap.add_argument("--valid-seqs", type=int, default=None, help="first N validation seqs '
                          '(default all)")\n'
                          '    ap.add_argument("--holdout", type=int, default=0, help="also score the N-seq train '
                          'holdout")\n'
                          '    ap.add_argument("--valid-split", help="split.json from mask_model: score validation '
                          'half B instead")\n'
                          '    ap.add_argument("--device", default="auto")\n'
                          '    args = ap.parse_args()\n'
                          '\n'
                          '    device = pick_device(args.device)\n'
                          '    net = (load_baseline("baseline/baseline.onnx") if args.baseline else '
                          'load_ckpt(args.ckpt)).to(device)\n'
                          '    out = {}\n'
                          '    t = time.time()\n'
                          '    valid = SeqStore("datasets/valid.parquet")\n'
                          '    ids = json.load(open(args.valid_split))["B"] if args.valid_split else '
                          'list(range(valid.n_seqs))\n'
                          '    out["valid_B" if args.valid_split else "valid"] = evaluate(net, valid, '
                          'ids[:args.valid_seqs or len(ids)], device)\n'
                          '    if args.holdout:\n'
                          '        train = SeqStore("datasets/train.parquet")\n'
                          '        _, hold = split_train(train.n_seqs, args.holdout)\n'
                          '        out["holdout"] = evaluate(net, train, hold, device)\n'
                          '    out["seconds"] = round(time.time() - t, 1)\n'
                          '    print(json.dumps(out, indent=1))\n'
                          '\n'
                          '\n'
                          'if __name__ == "__main__":\n'
                          '    main()\n',
 'pipeline/features.py': '"""Extra causal input features, computed identically offline (training, whole sequences) '
                         'and\n'
                         'online (inference, one row at a time):\n'
                         '\n'
                         "* activity of each instrument's prices / volumes / trades: mean |x_t - x_{t-1}| over the "
                         'group\n'
                         '  (0 on the first row of a sequence), now and as EMAs;\n'
                         '* EMAs of the extra features a2..a5 (a3 = the strongest scoring-mask signal).\n'
                         '\n'
                         'EMA_k(t) = EMA_k(t-1) + (x_t - EMA_k(t-1)) / span_k, started at x_0. All values standardised '
                         'with\n'
                         'fixed constants (MEAN / STD, measured once on training data). Width: N_EXTRA = 36.\n'
                         '\n'
                         'Feature sets ("kinds"): "act" = the 36 above; "norm" = every one of the 112 inputs as a '
                         'z-score against its\n'
                         'own running mean and std since the start of the sequence (per-sequence level and scale, '
                         'causal);\n'
                         '"act+norm" = both, in that order. A model\'s kind follows from its input width '
                         '(kind_for_width).\n'
                         'This file is copied next to solution.py as wnn_features.py, so it must only need numpy at '
                         'inference.\n'
                         '"""\n'
                         'import numpy as np\n'
                         '\n'
                         'GROUPS = [(0, 22), (22, 44), (44, 52), (52, 74), (74, 96), (96, 104)]  # i0 p, i0 v, i0 '
                         'trades, i1 p, i1 v, i1 trades\n'
                         'LEVEL_COLS = [106, 107, 108, 109]                                      # a2, a3, a4, a5\n'
                         'SPANS = [20, 200, 2000]\n'
                         'N_EXTRA = len(GROUPS) * (1 + len(SPANS)) + len(LEVEL_COLS) * len(SPANS)  # 36\n'
                         'NORM_FLOOR = 0.02  # added to the running std (inputs are globally standardised), so early '
                         'rows / flat columns stay finite\n'
                         'NORM_CLIP = 8.0\n'
                         'KINDS = {"act": N_EXTRA, "norm": 112, "act+norm": N_EXTRA + 112}\n'
                         '\n'
                         '\n'
                         'def width(kind):\n'
                         '    """Number of extra inputs of a feature kind (\'\' or None = none)."""\n'
                         '    return KINDS[kind] if kind else 0\n'
                         '\n'
                         '\n'
                         'def kind_for_width(n):\n'
                         '    return next(k for k, w in KINDS.items() if w == n)\n'
                         '\n'
                         '# standardisation constants: `python -m pipeline.features` on 256 training sequences\n'
                         'MEAN = [\n'
                         '    0.02899, 0.080176, 0.344639, 0.013477, 0.046691, 0.409072,\n'
                         '    0.028905, 0.079919, 0.344147, 0.013465, 0.046752, 0.408771,\n'
                         '    0.028743, 0.079457, 0.341971, 0.013384, 0.046461, 0.406134,\n'
                         '    0.026218, 0.07248, 0.31095, 0.012185, 0.04226, 0.36902,\n'
                         '    -0.197401, -0.235902, 0.050653, -0.37099, -0.197616, -0.236261,\n'
                         '    0.050583, -0.3714, -0.199425, -0.240088, 0.051767, -0.375679]\n'
                         'STD = [\n'
                         '    0.061435, 0.144017, 0.647244, 0.045508, 0.138009, 0.751662,\n'
                         '    0.035306, 0.083746, 0.311466, 0.019796, 0.072021, 0.43189,\n'
                         '    0.028954, 0.070935, 0.235276, 0.013908, 0.058679, 0.335773,\n'
                         '    0.02191, 0.058177, 0.200191, 0.010707, 0.048553, 0.27719,\n'
                         '    0.911831, 0.966895, 0.3914, 1.13453, 0.729865, 0.841434,\n'
                         '    0.244381, 0.879383, 0.598998, 0.688904, 0.264152, 0.809816]\n'
                         '\n'
                         '\n'
                         'def _raw_offline(X):\n'
                         '    """X [B,T,112] float32 -> unstandardised extras [B,T,36] float64 (whole sequences, exact '
                         'EMA)."""\n'
                         '    from scipy.signal import lfilter\n'
                         '\n'
                         '    X = X.astype(np.float64)\n'
                         '    d = np.abs(np.diff(X, axis=1, prepend=X[:, :1]))\n'
                         '    act = np.stack([d[..., a:b].mean(-1) for a, b in GROUPS], -1)      # [B,T,6]\n'
                         '    lev = X[..., LEVEL_COLS]                                           # [B,T,4]\n'
                         '    out = [act]\n'
                         '    for v in (act, lev):\n'
                         '        for span in SPANS:\n'
                         '            a = 1.0 / span\n'
                         '            zi = (1 - a) * v[:, :1]                                     # [B,1,F]: so that '
                         'EMA(0) = x_0\n'
                         '            y, _ = lfilter([a], [1, -(1 - a)], v, axis=1, zi=zi)\n'
                         '            out.append(y)\n'
                         '    # order: act, act_ema20, act_ema200, act_ema2000, lev_ema20, lev_ema200, lev_ema2000\n'
                         '    return np.concatenate(out, -1)\n'
                         '\n'
                         '\n'
                         'def _norm_offline(X):\n'
                         '    """X [B,T,112] -> running z-scores [B,T,112] float64. Values are taken relative to the '
                         'first row, so the\n'
                         '    float64 running sums stay exact enough; the online version adds them up in the same '
                         'order."""\n'
                         '    xs = X.astype(np.float64)\n'
                         '    xs -= xs[:, :1]\n'
                         '    n = np.arange(1, X.shape[1] + 1, dtype=np.float64)[None, :, None]\n'
                         '    mean = np.cumsum(xs, 1) / n\n'
                         '    var = np.maximum(np.cumsum(xs * xs, 1) / n - mean * mean, 0.0)\n'
                         '    return np.clip((xs - mean) / np.sqrt(var + NORM_FLOOR ** 2), -NORM_CLIP, NORM_CLIP)\n'
                         '\n'
                         '\n'
                         'def offline(X, kind="act", chunk=16):\n'
                         '    """Extras [B,T,width(kind)] float32 for whole sequences. Computed `chunk` sequences at a '
                         'time:\n'
                         '    the float64 intermediates of a whole 256-sequence batch would need ~10 GB of RAM."""\n'
                         '    out = np.empty(X.shape[:2] + (width(kind),), np.float32)\n'
                         '    for i in range(0, len(X), chunk):\n'
                         '        x = X[i:i + chunk]\n'
                         '        out[i:i + chunk] = np.concatenate([(_raw_offline(x) - MEAN) / STD if part == "act" '
                         'else _norm_offline(x)\n'
                         '                                           for part in kind.split("+")], -1)\n'
                         '    return out\n'
                         '\n'
                         '\n'
                         'class Online:\n'
                         '    """Row-by-row version of `offline(X, kind)`; reset() at every new sequence."""\n'
                         '\n'
                         '    def __init__(self, kind="act"):\n'
                         '        self.parts = [{"act": _ActOnline, "norm": _NormOnline}[p]() for p in '
                         'kind.split("+")]\n'
                         '\n'
                         '    def reset(self):\n'
                         '        for p in self.parts:\n'
                         '            p.reset()\n'
                         '\n'
                         '    def __call__(self, x):\n'
                         '        if len(self.parts) == 1:\n'
                         '            return self.parts[0](x)\n'
                         '        return np.concatenate([p(x) for p in self.parts])\n'
                         '\n'
                         '\n'
                         'class _NormOnline:\n'
                         '    def __init__(self):\n'
                         '        self.reset()\n'
                         '\n'
                         '    def reset(self):\n'
                         '        self.x0 = None\n'
                         '\n'
                         '    def __call__(self, x):\n'
                         '        xs = np.asarray(x, dtype=np.float64)\n'
                         '        if self.x0 is None:\n'
                         '            self.x0, self.n = xs.copy(), 0\n'
                         '            self.s1, self.s2 = np.zeros_like(xs), np.zeros_like(xs)\n'
                         '        xs = xs - self.x0\n'
                         '        self.n += 1\n'
                         '        self.s1 += xs\n'
                         '        self.s2 += xs * xs\n'
                         '        mean = self.s1 / self.n\n'
                         '        var = np.maximum(self.s2 / self.n - mean * mean, 0.0)\n'
                         '        return np.clip((xs - mean) / np.sqrt(var + NORM_FLOOR ** 2), -NORM_CLIP, '
                         'NORM_CLIP).astype(np.float32)\n'
                         '\n'
                         '\n'
                         'class _ActOnline:\n'
                         '    """Row-by-row "act" features. Kept to a handful of numpy calls per row: one matrix '
                         'product for all group\n'
                         '    activities, one array for all EMAs."""\n'
                         '\n'
                         '    def __init__(self):\n'
                         '        n_g, n_l = len(GROUPS), len(LEVEL_COLS)\n'
                         '        self.G = np.zeros((112, n_g))                                   # |dx| @ G = '
                         'per-group mean\n'
                         '        for k, (a, b) in enumerate(GROUPS):\n'
                         '            self.G[a:b, k] = 1.0 / (b - a)\n'
                         '        self.alpha = np.array([1.0 / s for s in SPANS])[:, None]       # [3,1]\n'
                         '        self.mean = np.asarray(MEAN, dtype=np.float64)\n'
                         '        self.inv_std = 1.0 / np.asarray(STD, dtype=np.float64)\n'
                         '        self.n_g = n_g\n'
                         '        self.raw = np.empty(N_EXTRA)\n'
                         '        self.lev_idx = np.array(LEVEL_COLS)\n'
                         '        self.reset()\n'
                         '\n'
                         '    def reset(self):\n'
                         '        self.prev = None\n'
                         '\n'
                         '    def __call__(self, x):\n'
                         '        x = np.asarray(x, dtype=np.float64)\n'
                         '        n_g = self.n_g\n'
                         '        if self.prev is None:\n'
                         '            act = np.zeros(n_g)\n'
                         '            self.ema = np.empty((len(SPANS), n_g + len(LEVEL_COLS)))     # [3, 6 activities '
                         '+ 4 levels]\n'
                         '            self.ema[:, :n_g] = 0.0\n'
                         '            self.ema[:, n_g:] = x[self.lev_idx]\n'
                         '        else:\n'
                         '            act = np.abs(x - self.prev) @ self.G\n'
                         '            v = np.concatenate((act, x[self.lev_idx]))\n'
                         '            self.ema += (v - self.ema) * self.alpha\n'
                         '        self.prev = x\n'
                         '        raw = self.raw\n'
                         '        raw[:n_g] = act\n'
                         '        raw[n_g:n_g * 4] = self.ema[:, :n_g].ravel()\n'
                         '        raw[n_g * 4:] = self.ema[:, n_g:].ravel()\n'
                         '        return ((raw - self.mean) * self.inv_std).astype(np.float32)\n'
                         '\n'
                         '\n'
                         'def compute_stats(n_seqs=256, data="datasets/train.parquet"):\n'
                         '    from .data import SeqStore\n'
                         '\n'
                         '    st = SeqStore(data)\n'
                         '    ids = list(range(0, st.n_seqs, st.n_seqs // n_seqs))[:n_seqs]\n'
                         '    R = []\n'
                         '    for i in range(0, len(ids), 32):\n'
                         '        X, _, _ = st.load(ids[i:i + 32])\n'
                         '        R.append(_raw_offline(X)[:, 99:].reshape(-1, N_EXTRA)[::7])\n'
                         '    R = np.concatenate(R)\n'
                         '    return R.mean(0), R.std(0) + 1e-6\n'
                         '\n'
                         '\n'
                         'if __name__ == "__main__":\n'
                         '    m, s = compute_stats()\n'
                         '    np.set_printoptions(precision=6, suppress=True, linewidth=150)\n'
                         '    print("MEAN = " + repr([round(float(v), 6) for v in m]))\n'
                         '    print("STD = " + repr([round(float(v), 6) for v in s]))\n',
 'pipeline/mask_model.py': '"""Scoring-mask model: predicts which rows the metric scores (is_scored) from causal '
                           'inputs only,\n'
                           'and turns it into per-row training weights.\n'
                           '\n'
                           '    python -m pipeline.mask_model fit   --out runs/mask_model          # train on '
                           'validation half A, AUC on half B\n'
                           '    python -m pipeline.mask_model score --model runs/mask_model --out mask_train.npy   # '
                           'p(scored) for train rows\n'
                           '\n'
                           "Validation is split once into halves A and B (split.json). The classifier sees only A's "
                           'masks, so\n'
                           'models can be compared on B without the mask information leaking into the comparison.\n'
                           'Output of `score`: float16 .npy [n_train_seqs, 20000], p(scored) per row (0 on warm-up '
                           'rows),\n'
                           'read by the trainer with Common.row_weights and ExpConfig.mask_weight.\n'
                           '"""\n'
                           'import argparse\n'
                           'import json\n'
                           'import os\n'
                           'import time\n'
                           '\n'
                           'import joblib\n'
                           'import numpy as np\n'
                           '\n'
                           'from utils import SEQUENCE_LENGTH, WARMUP\n'
                           '\n'
                           'from .data import SeqStore\n'
                           '\n'
                           'GROUPS = [slice(0, 11), slice(11, 22), slice(22, 33), slice(33, 44), slice(44, 48), '
                           'slice(48, 52),\n'
                           '          slice(52, 63), slice(63, 74), slice(74, 85), slice(85, 96), slice(96, 100), '
                           'slice(100, 104)]\n'
                           '\n'
                           '\n'
                           'def _trailing_mean(v, w):\n'
                           '    """Mean over the last min(w, t+1) rows, for every row t. v [B,T,F] -> [B,T,F]."""\n'
                           '    c = np.cumsum(v, 1, dtype=np.float64)\n'
                           '    c = np.concatenate([np.zeros_like(c[:, :1]), c], 1)\n'
                           '    t = np.arange(v.shape[1])\n'
                           '    lo = np.maximum(0, t + 1 - w)\n'
                           '    return ((c[:, t + 1] - c[:, lo]) / (t + 1 - lo)[None, :, None]).astype(np.float32)\n'
                           '\n'
                           '\n'
                           'def features(X):\n'
                           '    """Causal per-row features for rows WARMUP..T-1. X [B,T,112] -> [B, T-WARMUP, '
                           '142]."""\n'
                           '    d = np.abs(np.diff(X, axis=1, prepend=X[:, :1]))                     # |x_t - '
                           'x_{t-1}|, 0 at t=0\n'
                           '    dg = np.stack([d[..., g].mean(-1) for g in GROUPS], -1)               # activity per '
                           'book/trade group\n'
                           '    a = X[..., 106:109]                                                    # a2, a3, a4\n'
                           '    parts = [X, dg, _trailing_mean(a, 20), _trailing_mean(a, 200), _trailing_mean(dg, '
                           '20)]\n'
                           '    return np.concatenate(parts, -1)[:, WARMUP:]\n'
                           '\n'
                           '\n'
                           'def split_valid(n_seqs, seed=5):\n'
                           '    perm = np.random.default_rng(seed).permutation(n_seqs)\n'
                           '    half = n_seqs // 2\n'
                           '    return sorted(int(i) for i in perm[:half]), sorted(int(i) for i in perm[half:])\n'
                           '\n'
                           '\n'
                           'def predict_rows(model, X, chunk=8):\n'
                           '    """p(scored) [B, T] for a batch of sequences (0 on warm-up rows)."""\n'
                           '    out = np.zeros(X.shape[:2], np.float32)\n'
                           '    for i in range(0, len(X), chunk):\n'
                           '        f = features(X[i:i + chunk])\n'
                           '        out[i:i + chunk, WARMUP:] = model.predict_proba(f.reshape(-1, f.shape[-1]))[:, '
                           '1].reshape(f.shape[:2])\n'
                           '    return out\n'
                           '\n'
                           '\n'
                           'def fit(args):\n'
                           '    from sklearn.ensemble import HistGradientBoostingClassifier\n'
                           '    from sklearn.metrics import average_precision_score, roc_auc_score\n'
                           '\n'
                           '    rng = np.random.default_rng(0)\n'
                           '    st = SeqStore(os.path.join(args.data_dir, "valid.parquet"), args.threads)\n'
                           '    A, B = split_valid(st.n_seqs)\n'
                           '    os.makedirs(args.out, exist_ok=True)\n'
                           '    json.dump({"A": A, "B": B, "seed": 5}, open(os.path.join(args.out, "split.json"), '
                           '"w"))\n'
                           '\n'
                           '    t = time.time()\n'
                           '    F, Y = [], []\n'
                           '    for i in range(0, len(A), 32):  # sample rows of half A for training\n'
                           '        X, _, M = st.load(A[i:i + 32])\n'
                           '        f, m = features(X), M[:, WARMUP:]\n'
                           '        sel = rng.random(m.shape) < args.sample\n'
                           '        F.append(f[sel]); Y.append(m[sel])\n'
                           '    F, Y = np.concatenate(F), np.concatenate(Y)\n'
                           '    print(f"training rows {len(Y):,} from {len(A)} sequences (half A), load {time.time() - '
                           't:.0f}s", flush=True)\n'
                           '\n'
                           '    t = time.time()\n'
                           '    model = HistGradientBoostingClassifier(max_iter=args.trees, learning_rate=0.12, '
                           'max_leaf_nodes=63,\n'
                           '                                           early_stopping=False, random_state=0).fit(F, '
                           'Y)\n'
                           '    print(f"fit {time.time() - t:.0f}s", flush=True)\n'
                           '    joblib.dump(model, os.path.join(args.out, "model.joblib"))\n'
                           '\n'
                           '    # honest check on half B (sequences the classifier never saw)\n'
                           '    P, M = [], []\n'
                           '    for i in range(0, min(len(B), args.eval_seqs), 32):\n'
                           '        X, _, m = st.load(B[i:i + 32])\n'
                           '        P.append(predict_rows(model, X)[:, WARMUP:].ravel()); M.append(m[:, '
                           'WARMUP:].ravel())\n'
                           '    P, M = np.concatenate(P), np.concatenate(M)\n'
                           '    top10 = M[P >= np.quantile(P, 0.9)].sum() / M.sum()\n'
                           '    rep = {"auc_B": float(roc_auc_score(M, P)), "avg_precision_B": '
                           'float(average_precision_score(M, P)),\n'
                           '           "base_rate_B": float(M.mean()), "top10pct_share_of_scored": float(top10), '
                           '"eval_seqs": int(min(len(B), args.eval_seqs))}\n'
                           '    json.dump(rep, open(os.path.join(args.out, "report.json"), "w"), indent=1)\n'
                           '    print(json.dumps(rep, indent=1))\n'
                           '\n'
                           '\n'
                           'def score(args):\n'
                           '    model = joblib.load(os.path.join(args.model, "model.joblib"))\n'
                           '    st = SeqStore(os.path.join(args.data_dir, "train.parquet"), args.threads)\n'
                           '    n = args.limit or st.n_seqs\n'
                           '    out = np.lib.format.open_memmap(args.out, mode="w+", dtype=np.float16, '
                           'shape=(st.n_seqs, SEQUENCE_LENGTH))\n'
                           '    if args.limit:\n'
                           '        out[n:] = 0\n'
                           '    t, done = time.time(), 0\n'
                           '    for X, _, _ in st.batches(list(range(n)), 64):\n'
                           '        out[done:done + len(X)] = predict_rows(model, X).astype(np.float16)\n'
                           '        done += len(X)\n'
                           '        if done % 640 == 0 or done == n:\n'
                           '            print(f"{done}/{n} sequences, {time.time() - t:.0f}s", flush=True)\n'
                           '    out.flush()\n'
                           '    print(f"mean p(scored) on required rows: {float(np.asarray(out[:n:50, WARMUP:], '
                           'np.float32).mean()):.4f} -> {args.out}")\n'
                           '\n'
                           '\n'
                           'def main():\n'
                           '    ap = argparse.ArgumentParser()\n'
                           '    ap.add_argument("cmd", choices=["fit", "score"])\n'
                           '    ap.add_argument("--out", required=True)\n'
                           '    ap.add_argument("--model", help="folder written by fit (for score)")\n'
                           '    ap.add_argument("--data-dir", default="datasets")\n'
                           '    ap.add_argument("--sample", type=float, default=0.1, help="fraction of half-A rows '
                           'used for fitting")\n'
                           '    ap.add_argument("--trees", type=int, default=150)\n'
                           '    ap.add_argument("--eval-seqs", type=int, default=400)\n'
                           '    ap.add_argument("--limit", type=int, default=None, help="score only the first N train '
                           'sequences (tests)")\n'
                           '    ap.add_argument("--threads", type=int, default=8)\n'
                           '    args = ap.parse_args()\n'
                           '    fit(args) if args.cmd == "fit" else score(args)\n'
                           '\n'
                           '\n'
                           'if __name__ == "__main__":\n'
                           '    main()\n',
 'pipeline/model.py': '"""Stateful sequence models (GRU/LSTM, minGRU, causal TCN), metric-aligned losses, ONNX step '
                      'export and\n'
                      'baseline import. Every model has forward(x, h, x_prev) -> (pred, new_state) for chunked '
                      'training/evaluation;\n'
                      'minGRU and TCN also have stream_step() for the one-row ONNX graph (see '
                      'research/architectures_cnn_rnn_transformer.md)."""\n'
                      'import math\n'
                      '\n'
                      'import numpy as np\n'
                      'import torch\n'
                      'import torch.nn.functional as F\n'
                      'from torch import nn\n'
                      '\n'
                      'from .features import kind_for_width\n'
                      '\n'
                      'N_FEATURES = 112\n'
                      'CLIP = 2.0\n'
                      '\n'
                      '\n'
                      'def _model_inputs(x, x_prev, use_diff, n_extra):\n'
                      '    """The 112 features, then their row-to-row diffs (use_diff), then the extra causal features '
                      '(n_extra)."""\n'
                      '    base = x[..., :N_FEATURES]\n'
                      '    parts = [base]\n'
                      '    if use_diff:\n'
                      '        first = base[:, :1] if x_prev is None else x_prev[:, None, :N_FEATURES]\n'
                      '        prev = torch.cat([first, base[:, :-1]], 1)\n'
                      '        parts.append(base - prev)\n'
                      '    if n_extra:\n'
                      '        assert x.shape[-1] >= N_FEATURES + n_extra, "model needs the extra features as input"\n'
                      '        parts.append(x[..., N_FEATURES:N_FEATURES + n_extra])\n'
                      '    return torch.cat(parts, -1) if len(parts) > 1 else base\n'
                      '\n'
                      '\n'
                      'class GRUNet(nn.Module):\n'
                      '    """Stacked GRU (or LSTM with cell="lstm") + linear head. The recurrent module keeps the\n'
                      '    attribute name `gru` for either cell so older checkpoints load unchanged.\n'
                      '    n_aux > 0 adds a training-only head for auxiliary targets (trainer.aux_targets) on the same '
                      'hidden states:\n'
                      '    forward() keeps the last top-layer output in `last_out` for it; the exported graph only has '
                      '`head`."""\n'
                      '\n'
                      '    def __init__(self, hidden=128, layers=2, use_diff=False, dropout=0.0, cell="gru", '
                      'n_extra=0, feat_kind=None,\n'
                      '                 n_aux=0):\n'
                      '        super().__init__()\n'
                      '        self.config = dict(hidden=hidden, layers=layers, use_diff=use_diff, dropout=dropout, '
                      'cell=cell, n_extra=n_extra)\n'
                      '        self.config.update({k: v for k, v in dict(feat_kind=feat_kind, n_aux=n_aux).items() if '
                      'v})\n'
                      '        self.use_diff = use_diff\n'
                      '        self.cell = cell\n'
                      '        self.n_extra = n_extra  # extra causal features (pipeline/features.py), appended after '
                      'the 112 inputs\n'
                      '        self.feat_kind = feat_kind or (kind_for_width(n_extra) if n_extra else None)\n'
                      '        in_dim = N_FEATURES * (2 if use_diff else 1) + n_extra\n'
                      '        rnn = {"gru": nn.GRU, "lstm": nn.LSTM}[cell]\n'
                      '        self.gru = rnn(in_dim, hidden, layers, batch_first=True, dropout=dropout if layers > 1 '
                      'else 0.0)\n'
                      '        self.head = nn.Linear(hidden, 2)\n'
                      '        self.aux_head = nn.Linear(hidden, n_aux) if n_aux else None\n'
                      '        self.last_out = None\n'
                      '\n'
                      '    def forward(self, x, h=None, x_prev=None):\n'
                      '        """x [B,T,112 (+ extras)]; h [L,B,H] or None; x_prev [B,>=112] = last row of the '
                      'previous\n'
                      '        chunk (None at sequence start, where the first diff is zero). Input order to the GRU:\n'
                      '        the 112 features, then their diffs (use_diff), then the extras (n_extra)."""\n'
                      '        base = x[..., :N_FEATURES]\n'
                      '        parts = [base]\n'
                      '        if self.use_diff:\n'
                      '            first = base[:, :1] if x_prev is None else x_prev[:, None, :N_FEATURES]\n'
                      '            prev = torch.cat([first, base[:, :-1]], 1)\n'
                      '            parts.append(base - prev)\n'
                      '        if self.n_extra:\n'
                      '            assert x.shape[-1] >= N_FEATURES + self.n_extra, "model needs the extra features as '
                      'input"\n'
                      '            parts.append(x[..., N_FEATURES:N_FEATURES + self.n_extra])\n'
                      '        out, h = self.gru(torch.cat(parts, -1) if len(parts) > 1 else base, h)\n'
                      '        if self.aux_head is not None:\n'
                      '            self.last_out = out\n'
                      '        return self.head(out), h\n'
                      '\n'
                      '\n'
                      'def _g(x):\n'
                      '    """minGRU candidate activation: x + 0.5 for x >= 0, sigmoid(x) below, so the candidate is '
                      'always positive."""\n'
                      '    return torch.where(x >= 0, x + 0.5, torch.sigmoid(x))\n'
                      '\n'
                      '\n'
                      'def _log_g(x):\n'
                      '    return torch.where(x >= 0, torch.log(F.relu(x) + 0.5), -F.softplus(-x))\n'
                      '\n'
                      '\n'
                      'class MinGRUNet(nn.Module):\n'
                      '    """Stack of minGRU layers (Feng et al. 2024, "Were RNNs All We Needed?") + linear head.\n'
                      '\n'
                      '    Per layer: z_t = sigmoid(W_z u_t), h_t = (1 - z_t) * h_{t-1} + z_t * g(W_h u_t). Gate and '
                      'candidate depend\n'
                      '    only on the layer input u_t (not on h_{t-1}), so training runs a parallel log-space scan '
                      'over each chunk\n'
                      '    (no step-by-step loop; long chunks are cheap), while inference stays an O(1) update per row '
                      'with ~1/3 of a GRU\n'
                      "    layer's multiply-adds. The nonlinear state interactions a GRU has come from stacking layers "
                      'instead.\n'
                      '    Gate biases start at log-spaced memory lengths of 2..2000 rows (chrono-style), so some '
                      'units remember long\n'
                      '    from the first step. State: one [B, H] tensor per layer."""\n'
                      '    BLOCK = 128  # scan block: keeps cumulative log-gates small enough for float32 (state is '
                      'carried across blocks)\n'
                      '\n'
                      '    def __init__(self, hidden=192, layers=3, use_diff=False, dropout=0.0, n_extra=0, '
                      'feat_kind=None):\n'
                      '        super().__init__()\n'
                      '        self.config = dict(arch="mingru", hidden=hidden, layers=layers, use_diff=use_diff, '
                      'dropout=dropout,\n'
                      '                           n_extra=n_extra, **({"feat_kind": feat_kind} if feat_kind else {}))\n'
                      '        self.use_diff, self.n_extra, self.hidden = use_diff, n_extra, hidden\n'
                      '        self.feat_kind = feat_kind or (kind_for_width(n_extra) if n_extra else None)\n'
                      '        in_dim = N_FEATURES * (2 if use_diff else 1) + n_extra\n'
                      '        self.proj = nn.ModuleList([nn.Linear(in_dim if l == 0 else hidden, 2 * hidden) for l in '
                      'range(layers)])\n'
                      '        self.drop = nn.Dropout(dropout)\n'
                      '        self.head = nn.Linear(hidden, 2)\n'
                      '        with torch.no_grad():  # chrono init: 1 / z = memory length tau, log-uniform in [2, '
                      '2000]\n'
                      '            for p in self.proj:\n'
                      '                tau = torch.exp(torch.empty(hidden).uniform_(math.log(2.0), math.log(2000.0)))\n'
                      '                p.bias[:hidden] = -torch.log(tau - 1)\n'
                      '\n'
                      '    @classmethod\n'
                      '    def _scan(cls, k, c, h0):\n'
                      '        """h_t = a_t h_{t-1} + b_t with a = 1 - sigmoid(k), b = sigmoid(k) g(c), in log space '
                      'per block:\n'
                      '        h_t = A_t h0 + sum_{j<=t} (A_t / A_j) b_j, A_t = prod_{i<=t} a_i. k, c [B,T,H]; h0 '
                      '[B,H] or None (zeros)."""\n'
                      '        log_a = -F.softplus(k)\n'
                      '        log_b = -F.softplus(-k) + _log_g(c)\n'
                      '        outs = []\n'
                      '        for s in range(0, k.shape[1], cls.BLOCK):\n'
                      '            a_star = log_a[:, s:s + cls.BLOCK].cumsum(1)\n'
                      '            h = torch.exp(a_star + torch.logcumsumexp(log_b[:, s:s + cls.BLOCK] - a_star, 1))\n'
                      '            if h0 is not None:\n'
                      '                h = h + torch.exp(a_star) * h0[:, None]\n'
                      '            outs.append(h)\n'
                      '            h0 = h[:, -1]\n'
                      '        return torch.cat(outs, 1)\n'
                      '\n'
                      '    def forward(self, x, h=None, x_prev=None):\n'
                      '        """x [B,T,112 (+ extras)]; h list of [B,H] per layer or None; x_prev as in GRUNet."""\n'
                      '        z = _model_inputs(x, x_prev, self.use_diff, self.n_extra)\n'
                      '        new = []\n'
                      '        for l, proj in enumerate(self.proj):\n'
                      '            k, c = proj(z).chunk(2, -1)\n'
                      '            z = self._scan(k, c, None if h is None else h[l])\n'
                      '            new.append(z[:, -1])\n'
                      '            if l < len(self.proj) - 1:\n'
                      '                z = self.drop(z)\n'
                      '        return self.head(z), new\n'
                      '\n'
                      '    def stream_states(self):\n'
                      '        return [(f"h{l}", (1, self.hidden)) for l in range(len(self.proj))]\n'
                      '\n'
                      '    def stream_step(self, z, states):\n'
                      '        """One row: z [1,1,in] (already built inputs), states list of [1,H] -> (pred [1,1,2], '
                      'new states)."""\n'
                      '        z, new = z[:, 0], []\n'
                      '        for proj, h in zip(self.proj, states):\n'
                      '            k, c = proj(z).chunk(2, -1)\n'
                      '            z = h + torch.sigmoid(k) * (_g(c) - h)\n'
                      '            new.append(z)\n'
                      '        return self.head(z)[:, None], new\n'
                      '\n'
                      '\n'
                      'class TCNNet(nn.Module):\n'
                      '    """Causal dilated temporal convolution network (Bai et al. 2018, WaveNet-style) + linear '
                      'head.\n'
                      '\n'
                      '    Input projection to `hidden` channels, then `layers` pre-norm residual blocks\n'
                      '    z <- z + W_mix relu(conv_{kernel, dilation}(LayerNorm(z))), dilation = kernel**l, so the '
                      'receptive field\n'
                      '    is kernel**layers rows (2**11 = 2048, 3**7 = 2187). Each output sees a fixed window of past '
                      'rows; unlike an\n'
                      '    RNN nothing older can leak in. State between chunks: the last (kernel - 1) * dilation '
                      'normalised inputs of\n'
                      '    each block ([B, pad_l, C]), zeros before the sequence start, so chunked training, '
                      'evaluation and row-by-row\n'
                      '    inference see exactly the same context."""\n'
                      '\n'
                      '    def __init__(self, hidden=64, layers=11, use_diff=False, dropout=0.0, n_extra=0, kernel=2, '
                      'feat_kind=None):\n'
                      '        super().__init__()\n'
                      '        assert kernel >= 2, "TCN kernel must be >= 2"\n'
                      '        self.config = dict(arch="tcn", hidden=hidden, layers=layers, use_diff=use_diff, '
                      'dropout=dropout,\n'
                      '                           n_extra=n_extra, kernel=kernel, **({"feat_kind": feat_kind} if '
                      'feat_kind else {}))\n'
                      '        self.use_diff, self.n_extra, self.hidden, self.kernel = use_diff, n_extra, hidden, '
                      'kernel\n'
                      '        self.feat_kind = feat_kind or (kind_for_width(n_extra) if n_extra else None)\n'
                      '        self.dilations = [kernel ** l for l in range(layers)]\n'
                      '        self.pads = [(kernel - 1) * d for d in self.dilations]\n'
                      '        in_dim = N_FEATURES * (2 if use_diff else 1) + n_extra\n'
                      '        self.inp = nn.Linear(in_dim, hidden)\n'
                      '        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(layers)])\n'
                      '        self.convs = nn.ModuleList([nn.Conv1d(hidden, hidden, kernel, dilation=d) for d in '
                      'self.dilations])\n'
                      '        self.mix = nn.ModuleList([nn.Linear(hidden, hidden) for _ in range(layers)])\n'
                      '        self.drop = nn.Dropout(dropout)\n'
                      '        self.out_norm = nn.LayerNorm(hidden)\n'
                      '        self.head = nn.Linear(hidden, 2)\n'
                      '\n'
                      '    @property\n'
                      '    def receptive_field(self):\n'
                      '        return 1 + sum(self.pads)\n'
                      '\n'
                      '    def forward(self, x, h=None, x_prev=None):\n'
                      '        """x [B,T,112 (+ extras)]; h list of [B, pad_l, C] per block or None; x_prev as in '
                      'GRUNet."""\n'
                      '        z = self.inp(_model_inputs(x, x_prev, self.use_diff, self.n_extra))\n'
                      '        B, new = z.shape[0], []\n'
                      '        for l, (norm, conv, mix, pad) in enumerate(zip(self.norms, self.convs, self.mix, '
                      'self.pads)):\n'
                      '            u = norm(z)\n'
                      '            past = h[l] if h is not None else u.new_zeros(B, pad, u.shape[-1])\n'
                      '            full = torch.cat([past, u], 1)  # [B, pad + T, C]\n'
                      '            new.append(full[:, -pad:])\n'
                      '            y = conv(full.transpose(1, 2)).transpose(1, 2)  # [B, T, C]\n'
                      '            z = z + mix(self.drop(F.relu(y)))\n'
                      '        return self.head(self.out_norm(z)), new\n'
                      '\n'
                      '    def stream_states(self):\n'
                      '        return [(f"b{l}", (1, pad, self.hidden)) for l, pad in enumerate(self.pads)]\n'
                      '\n'
                      '    def stream_step(self, z, states):\n'
                      '        """One row: z [1,1,in], states list of [1, pad_l, C] -> (pred [1,1,2], new states). '
                      'Buffers are shifted\n'
                      '        with Concat + Slice (simple, but copies every buffer once per row: see the notes on '
                      'ring buffers)."""\n'
                      '        z, new = self.inp(z), []\n'
                      '        for norm, conv, mix, d, buf in zip(self.norms, self.convs, self.mix, self.dilations, '
                      'states):\n'
                      '            full = torch.cat([buf, norm(z)], 1)  # [1, pad + 1, C]\n'
                      '            new.append(full[:, 1:])\n'
                      "            taps = full[:, ::d]  # the kernel's inputs: rows t - (k-1)d, ..., t - d, t\n"
                      '            y = F.conv1d(taps.transpose(1, 2), conv.weight, conv.bias).transpose(1, 2)  # [1, '
                      '1, C]\n'
                      '            z = z + mix(F.relu(y))\n'
                      '        return self.head(self.out_norm(z)), new\n'
                      '\n'
                      '\n'
                      'ARCHS = {"gru": GRUNet, "mingru": MinGRUNet, "tcn": TCNNet}\n'
                      '\n'
                      '\n'
                      'def build_net(config):\n'
                      '    """Model from a saved config. GRU/LSTM configs predate the `arch` key, so it defaults to '
                      'gru."""\n'
                      '    cfg = dict(config)\n'
                      '    return ARCHS[cfg.pop("arch", "gru")](**cfg)\n'
                      '\n'
                      '\n'
                      'def needs_extra(net):\n'
                      '    members = list(net.nets) if isinstance(net, EnsembleNet) else [net]\n'
                      '    return max(getattr(m, "n_extra", 0) for m in members)\n'
                      '\n'
                      '\n'
                      'def feat_kind(net):\n'
                      '    """The feature kind a model (or every member of an ensemble) takes as extra inputs; None = '
                      'plain 112."""\n'
                      '    members = list(net.nets) if isinstance(net, EnsembleNet) else [net]\n'
                      '    kinds = {getattr(m, "feat_kind", None) for m in members}\n'
                      '    assert len(kinds - {None}) <= 1, f"members use different feature kinds: {kinds}"\n'
                      '    return max(kinds - {None}, default=None)\n'
                      '\n'
                      '\n'
                      'def detach_state(h):\n'
                      '    """Detach a recurrent state: tensor (GRU), (h, c) tuple (LSTM) or list of those '
                      '(ensemble)."""\n'
                      '    if isinstance(h, torch.Tensor):\n'
                      '        return h.detach()\n'
                      '    return type(h)(detach_state(x) for x in h)\n'
                      '\n'
                      '\n'
                      'def macs(config) -> int:\n'
                      '    """Multiply-adds per row (inference cost model). Ensembles: sum over members."""\n'
                      '    if "members" in config:\n'
                      '        return sum(macs(m) for m in config["members"])\n'
                      '    arch = config.get("arch", "gru")\n'
                      '    H, n_in = config["hidden"], N_FEATURES * (2 if config["use_diff"] else 1) + '
                      'config.get("n_extra", 0)\n'
                      '    if arch == "mingru":  # one [in -> 2H] projection per layer\n'
                      '        return sum(2 * H * (n_in if l == 0 else H) for l in range(config["layers"])) + 2 * H\n'
                      '    if arch == "tcn":  # input projection, per block a k-tap conv + 1x1 mix (LayerNorms '
                      'ignored)\n'
                      '        return n_in * H + config["layers"] * (config["kernel"] + 1) * H * H + 2 * H\n'
                      '    gates = 4 if config.get("cell", "gru") == "lstm" else 3\n'
                      '    total = 0\n'
                      '    for _ in range(config["layers"]):\n'
                      '        total += gates * H * (n_in + H)\n'
                      '        n_in = H\n'
                      '    return total + 2 * H\n'
                      '\n'
                      '\n'
                      'BASELINE_MACS = macs(dict(hidden=128, layers=2, use_diff=False))  # 190,720\n'
                      '\n'
                      '\n'
                      'def _rw(w, rw):\n'
                      '    """Multiply metric weights [N,2] by optional per-row weights [N]."""\n'
                      '    return w if rw is None else w * rw[:, None]\n'
                      '\n'
                      '\n'
                      'def wmse_loss(p, y, rw=None):\n'
                      '    """Weighted MSE with the metric\'s clipping and |y| weights (times optional row weights). '
                      'p, y: [N, 2]."""\n'
                      '    y = y.clamp(-CLIP, CLIP)\n'
                      '    w = _rw(y.abs(), rw)\n'
                      '    return ((w * (p - y) ** 2).sum(0) / (w.sum(0) + 1e-8)).mean()\n'
                      '\n'
                      '\n'
                      'def distill_loss(p, teacher, y, weight="y", rw=None):\n'
                      '    """MSE to the teacher\'s predictions. Row weights: "y" = |clipped target| (like the '
                      'metric),\n'
                      '    "none" = uniform, "teacher" = |teacher prediction| (no dependence on the noisy target)."""\n'
                      '    if weight == "y":\n'
                      '        w = y.clamp(-CLIP, CLIP).abs()\n'
                      '    elif weight == "teacher":\n'
                      '        w = teacher.abs()\n'
                      '    elif weight == "none":\n'
                      '        w = torch.ones_like(teacher)\n'
                      '    else:\n'
                      '        raise ValueError(weight)\n'
                      '    w = _rw(w, rw)\n'
                      '    return ((w * (p - teacher) ** 2).sum(0) / (w.sum(0) + 1e-8)).mean()\n'
                      '\n'
                      '\n'
                      'def wpearson(p, y, rw=None):\n'
                      '    """Per-target weighted Pearson (pooled over the rows given), metric conventions. -> '
                      '[2]."""\n'
                      '    y = y.clamp(-CLIP, CLIP)\n'
                      '    w = _rw(y.abs(), rw)\n'
                      '    W = w.sum(0) + 1e-8\n'
                      '    yc = y - (w * y).sum(0) / W\n'
                      '    pc = p - (w * p).sum(0) / W\n'
                      '    cov = (w * yc * pc).sum(0)\n'
                      '    return cov / torch.sqrt((w * yc * yc).sum(0) * (w * pc * pc).sum(0) + 1e-12)\n'
                      '\n'
                      '\n'
                      'def compute_loss(kind, p, y, alpha=0.1, rw=None):\n'
                      '    if kind == "wmse":\n'
                      '        return wmse_loss(p, y, rw)\n'
                      '    if kind == "pearson":\n'
                      '        return 1 - wpearson(p, y, rw).mean()\n'
                      '    if kind == "mix":  # correlation objective, small wmse term anchors the scale for clipping\n'
                      '        return 1 - wpearson(p, y, rw).mean() + alpha * wmse_loss(p, y, rw)\n'
                      '    raise ValueError(kind)\n'
                      '\n'
                      '\n'
                      'class EnsembleNet(nn.Module):\n'
                      '    """Average of several GRUNets. Each member\'s output is multiplied by `scales[k]` (per '
                      'target)\n'
                      '    first, so members with different prediction spreads count equally. h is a list, one per '
                      'member."""\n'
                      '\n'
                      '    def __init__(self, nets, scales):\n'
                      '        super().__init__()\n'
                      '        self.nets = nn.ModuleList(nets)\n'
                      '        self.register_buffer("scales", torch.as_tensor(np.asarray(scales), '
                      'dtype=torch.float32))  # [K, 2]\n'
                      '        self.use_diff = any(n.use_diff for n in nets)\n'
                      '        self.config = {"members": [n.config for n in nets], "scales": '
                      'np.asarray(scales).tolist()}\n'
                      '\n'
                      '    def forward(self, x, h=None, x_prev=None):\n'
                      '        h = h if h is not None else [None] * len(self.nets)\n'
                      '        preds, new_h = [], []\n'
                      '        for k, (net, hk) in enumerate(zip(self.nets, h)):\n'
                      '            p, hn = net(x, hk, x_prev if net.use_diff else None)\n'
                      '            preds.append(p * self.scales[k])\n'
                      '            new_h.append(hn)\n'
                      '        return torch.stack(preds).mean(0), new_h\n'
                      '\n'
                      '\n'
                      'def _split_layers(net):\n'
                      '    """Single-layer copies of a stacked GRU/LSTM, so the ONNX graph gets one state input per '
                      'layer\n'
                      '    (as in the baseline) instead of slicing/concatenating a stacked state on every row."""\n'
                      '    rnn, mods = net.gru, []\n'
                      '    for l in range(rnn.num_layers):\n'
                      '        m = type(rnn)(rnn.input_size if l == 0 else rnn.hidden_size, rnn.hidden_size, 1, '
                      'batch_first=True)\n'
                      '        for n in ("weight_ih", "weight_hh", "bias_ih", "bias_hh"):\n'
                      '            getattr(m, f"{n}_l0").data.copy_(getattr(rnn, f"{n}_l{l}").data)\n'
                      '        mods.append(m)\n'
                      '    return nn.ModuleList(mods)\n'
                      '\n'
                      '\n'
                      'def _member_states(m):\n'
                      '    """(name suffix, shape) of each streaming state of one model, in graph order."""\n'
                      '    if isinstance(m, GRUNet):\n'
                      '        H, lstm = m.config["hidden"], m.cell == "lstm"\n'
                      '        return [s for l in range(m.config["layers"])\n'
                      '                for s in [(f"h{l}", (1, 1, H))] + ([(f"c{l}", (1, 1, H))] if lstm else [])]\n'
                      '    return m.stream_states()\n'
                      '\n'
                      '\n'
                      'class _Step(nn.Module):\n'
                      '    """One-row step graph: (x [1,1,112], x_prev [1,112], *states) -> (pred [1,1,2], *new '
                      'states).\n'
                      '    States are flat, per member: GRU one h [1,1,H] per layer, LSTM h and c per layer, minGRU '
                      'one h [1,H] per\n'
                      '    layer, TCN one input buffer [1, pad, C] per block."""\n'
                      '\n'
                      '    def __init__(self, net):\n'
                      '        super().__init__()\n'
                      '        self.ensemble = isinstance(net, EnsembleNet)\n'
                      '        members = list(net.nets) if self.ensemble else [net]\n'
                      '        self.rnn = [isinstance(m, GRUNet) for m in members]\n'
                      '        self.layers = nn.ModuleList([_split_layers(m) if r else nn.ModuleList() for m, r in '
                      'zip(members, self.rnn)])\n'
                      '        self.heads = nn.ModuleList([m.head if r else nn.Identity() for m, r in zip(members, '
                      'self.rnn)])\n'
                      '        self.streams = nn.ModuleList([nn.Identity() if r else m for m, r in zip(members, '
                      'self.rnn)])\n'
                      '        self.n_states = [len(_member_states(m)) for m in members]\n'
                      '        self.diff = [m.use_diff for m in members]\n'
                      '        self.n_extra = [getattr(m, "n_extra", 0) for m in members]\n'
                      '        self.lstm = [r and m.cell == "lstm" for m, r in zip(members, self.rnn)]\n'
                      '        self.register_buffer("scales", net.scales if self.ensemble else torch.ones(1, 2))\n'
                      '\n'
                      '    def forward(self, x, x_prev, *states):\n'
                      '        states, new, preds, i = list(states), [], [], 0\n'
                      '        for k, layers in enumerate(self.layers):\n'
                      '            base = x[..., :N_FEATURES]\n'
                      '            parts = [base] + ([base - x_prev[:, None]] if self.diff[k] else []) + \\\n'
                      '                    ([x[..., N_FEATURES:N_FEATURES + self.n_extra[k]]] if self.n_extra[k] else '
                      '[])\n'
                      '            z = torch.cat(parts, -1) if len(parts) > 1 else base\n'
                      "            if not self.rnn[k]:  # minGRU / TCN: the model's own one-row update\n"
                      '                p, ns = self.streams[k].stream_step(z, states[i:i + self.n_states[k]])\n'
                      '                new += ns\n'
                      '                i += self.n_states[k]\n'
                      '            else:\n'
                      '                for layer in layers:\n'
                      '                    if self.lstm[k]:\n'
                      '                        z, (h, c) = layer(z, (states[i], states[i + 1]))\n'
                      '                        new += [h, c]\n'
                      '                        i += 2\n'
                      '                    else:\n'
                      '                        z, h = layer(z, states[i])\n'
                      '                        new.append(h)\n'
                      '                        i += 1\n'
                      '                p = self.heads[k](z)\n'
                      '            preds.append(p * self.scales[k] if self.ensemble else p)\n'
                      '        pred = preds[0] if len(preds) == 1 else torch.stack(preds).mean(0)\n'
                      '        return (pred, *new)\n'
                      '\n'
                      '\n'
                      'def state_names(net):\n'
                      '    members = list(net.nets) if isinstance(net, EnsembleNet) else [net]\n'
                      '    return [f"m{k}_{s}" for k, m in enumerate(members) for s, _ in _member_states(m)]\n'
                      '\n'
                      '\n'
                      'def export_onnx(net, path: str):\n'
                      '    """Export a model (GRUNet, MinGRUNet, TCNNet) or EnsembleNet as a one-row step graph. '
                      'Inputs: x, x_prev\n'
                      '    (dropped from the graph when no member uses diff features), then the states per member '
                      'named m{k}_<state>\n'
                      '    (h{l} / c{l} for GRU/LSTM and minGRU, b{l} for TCN buffers); outputs: pred, then the new '
                      'states in the same\n'
                      "    order. All states start at zero (solution.py zero-fills them from the graph's input "
                      'shapes)."""\n'
                      '    net = net.eval().cpu()\n'
                      '    feat_kind(net)  # raises if ensemble members need different extra features\n'
                      '    names = state_names(net)\n'
                      '    members = list(net.nets) if isinstance(net, EnsembleNet) else [net]\n'
                      '    states = [torch.zeros(*shape) for m in members for _, shape in _member_states(m)]\n'
                      '    args = (torch.zeros(1, 1, N_FEATURES + needs_extra(net)), torch.zeros(1, N_FEATURES), '
                      '*states)\n'
                      '    with torch.no_grad():\n'
                      '        torch.onnx.export(_Step(net), args, path, input_names=["x", "x_prev", *names],\n'
                      '                          output_names=["pred", *[n + "_out" for n in names]], '
                      'opset_version=17, dynamo=False)\n'
                      '\n'
                      '\n'
                      'def load_baseline(onnx_path: str) -> GRUNet:\n'
                      '    """Load the organisers\' 2-layer GRU ONNX weights into a GRUNet (for checks / warm '
                      'starts)."""\n'
                      '    import onnx\n'
                      '    from onnx import numpy_helper\n'
                      '\n'
                      '    g = onnx.load(onnx_path).graph\n'
                      '    init = {t.name: numpy_helper.to_array(t) for t in g.initializer}\n'
                      '    grus = [n for n in g.node if n.op_type == "GRU"]\n'
                      '    matmul = next(n for n in g.node if n.op_type == "MatMul")\n'
                      '    H = 128\n'
                      '    net = GRUNet(hidden=H, layers=len(grus))\n'
                      '\n'
                      '    def reorder(a):  # ONNX gates z,r,h -> PyTorch r,z,n\n'
                      '        return np.concatenate([a[H:2 * H], a[:H], a[2 * H:]], 0)\n'
                      '\n'
                      '    sd = {}\n'
                      '    for l, n in enumerate(grus):\n'
                      '        W, R, B = (init[n.input[i]][0] for i in (1, 2, 3))\n'
                      '        sd[f"gru.weight_ih_l{l}"] = reorder(W)\n'
                      '        sd[f"gru.weight_hh_l{l}"] = reorder(R)\n'
                      '        sd[f"gru.bias_ih_l{l}"] = reorder(B[:3 * H])\n'
                      '        sd[f"gru.bias_hh_l{l}"] = reorder(B[3 * H:])\n'
                      '    sd["head.weight"] = init[matmul.input[1]].T\n'
                      '    sd["head.bias"] = init["reg_head.bias"]\n'
                      '    net.load_state_dict({k: torch.from_numpy(np.array(v, dtype=np.float32)) for k, v in '
                      'sd.items()})\n'
                      '    return net\n',
 'pipeline/multi_train.py': '"""Train several experiments side by side from one data stream.\n'
                            '\n'
                            '    python -m pipeline.multi_train --config experiments.json --epochs 4 --threads 8 '
                            '--cache-eval\n'
                            '\n'
                            'experiments.json is a list of ExpConfig dicts, e.g.\n'
                            '    [{"name": "wmse"}, {"name": "mix", "loss": "mix"}, {"name": "diff_h192", "diff": '
                            'true, "hidden": 192}]\n'
                            'Shared settings (epochs, batch, chunk, data/out dirs, ...) are the Common fields, as '
                            'flags.\n'
                            '"""\n'
                            'import argparse\n'
                            'import dataclasses\n'
                            'import json\n'
                            'import typing\n'
                            '\n'
                            'from .trainer import Common, ExpConfig, train_many\n'
                            '\n'
                            '\n'
                            'def field_type(f):\n'
                            '    """int | None -> int, str | None -> str, plain types unchanged."""\n'
                            '    args = [a for a in typing.get_args(f.type) if a is not type(None)]\n'
                            '    return args[0] if args else f.type\n'
                            '\n'
                            '\n'
                            'def add_flags(ap, cls, skip=("extra",)):\n'
                            '    for f in dataclasses.fields(cls):\n'
                            '        if f.name in skip:\n'
                            '            continue\n'
                            '        flag = "--" + f.name.replace("_", "-")\n'
                            '        if field_type(f) is bool:\n'
                            '            ap.add_argument(flag, action="store_true")\n'
                            '        else:\n'
                            '            ap.add_argument(flag, type=field_type(f), default=f.default)\n'
                            '\n'
                            '\n'
                            'def add_common_flags(ap):\n'
                            '    add_flags(ap, Common)\n'
                            '\n'
                            '\n'
                            'def main():\n'
                            '    ap = argparse.ArgumentParser()\n'
                            '    ap.add_argument("--config", required=True, help="JSON list of experiment configs")\n'
                            '    add_common_flags(ap)\n'
                            '    args = vars(ap.parse_args())\n'
                            '    configs = [ExpConfig(**c) for c in json.load(open(args.pop("config")))]\n'
                            '    names = [c.name for c in configs]\n'
                            '    assert len(set(names)) == len(names), "experiment names must be unique"\n'
                            '    train_many(configs, Common(**args))\n'
                            '\n'
                            '\n'
                            'if __name__ == "__main__":\n'
                            '    main()\n',
 'pipeline/solution_template.py': '"""Generic stateful ONNX step runner. Copied as solution.py next to model.onnx by '
                                  'the pipeline.\n'
                                  'Works for single models and ensembles: every graph input other than x / x_prev is a '
                                  'recurrent\n'
                                  'state, and graph outputs[1:] are their updates in the same order."""\n'
                                  'from pathlib import Path\n'
                                  'import os\n'
                                  'import sys\n'
                                  '\n'
                                  'os.environ.setdefault("OMP_NUM_THREADS", "1")\n'
                                  'os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")\n'
                                  'os.environ.setdefault("MKL_NUM_THREADS", "1")\n'
                                  '\n'
                                  'import numpy as np\n'
                                  'import onnxruntime as ort\n'
                                  '\n'
                                  '\n'
                                  'class PredictionModel:\n'
                                  '    def __init__(self):\n'
                                  '        o = ort.SessionOptions()\n'
                                  '        o.intra_op_num_threads = 1\n'
                                  '        o.inter_op_num_threads = 1\n'
                                  '        o.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL\n'
                                  '        o.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL\n'
                                  '        o.add_session_config_entry("session.intra_op.allow_spinning", "0")\n'
                                  '        self.session = ort.InferenceSession(str(Path(__file__).resolve().parent / '
                                  '"model.onnx"),\n'
                                  '                                            sess_options=o, '
                                  'providers=["CPUExecutionProvider"])\n'
                                  '        inputs = self.session.get_inputs()\n'
                                  '        self.state_names = [i.name for i in inputs if i.name not in ("x", '
                                  '"x_prev")]\n'
                                  '        self.state_init = [np.zeros(i.shape, dtype=np.float32) for i in inputs if '
                                  'i.name in self.state_names]\n'
                                  '        self.use_prev = any(i.name == "x_prev" for i in inputs)  # absent when no '
                                  'model uses diff features\n'
                                  '        self.out_names = [o.name for o in self.session.get_outputs()]\n'
                                  '        x_width = next(i.shape[-1] for i in inputs if i.name == "x")\n'
                                  '        self.features = None\n'
                                  '        if x_width > 112:  # the model also takes the extra causal features '
                                  '(wnn_features.py, shipped alongside)\n'
                                  '            sys.path.insert(0, str(Path(__file__).resolve().parent))\n'
                                  '            import wnn_features\n'
                                  '            kinds = getattr(wnn_features, "KINDS", {"act": wnn_features.N_EXTRA})\n'
                                  '            kind = next((k for k, w in kinds.items() if 112 + w == x_width), None)\n'
                                  '            assert kind, "wnn_features.py does not match model.onnx"\n'
                                  '            self.features = wnn_features.Online(kind) if hasattr(wnn_features, '
                                  '"KINDS") else wnn_features.Online()\n'
                                  '        self.seq_ix = None\n'
                                  '\n'
                                  '    def _reset(self):\n'
                                  '        self.feeds = dict(zip(self.state_names, [s.copy() for s in '
                                  'self.state_init]))\n'
                                  '        self.x_prev = None\n'
                                  '        if self.features is not None:\n'
                                  '            self.features.reset()\n'
                                  '\n'
                                  '    def predict(self, data_point):\n'
                                  '        if data_point.seq_ix != self.seq_ix:\n'
                                  '            self.seq_ix = data_point.seq_ix\n'
                                  '            self._reset()\n'
                                  '        feeds = self.feeds\n'
                                  '        state = np.asarray(data_point.state, dtype=np.float32)\n'
                                  '        x = state.reshape(1, 1, 112)\n'
                                  '        if self.use_prev:\n'
                                  '            feeds["x_prev"] = x[0] if self.x_prev is None else self.x_prev\n'
                                  '            self.x_prev = x[0]\n'
                                  '        if self.features is not None:\n'
                                  '            x = np.concatenate((state, self.features(state))).reshape(1, 1, -1)\n'
                                  '        feeds["x"] = x\n'
                                  '        out = self.session.run(self.out_names, feeds)\n'
                                  '        for name, value in zip(self.state_names, out[1:]):\n'
                                  '            feeds[name] = value\n'
                                  '        pred = out[0].reshape(2)\n'
                                  '        if not np.isfinite(pred).all():  # never seen; keep the submission valid '
                                  'and restart the state\n'
                                  '            self._reset()\n'
                                  '            pred = np.zeros(2, dtype=np.float32)\n'
                                  '        if not data_point.need_prediction:\n'
                                  '            return None\n'
                                  '        return pred\n',
 'pipeline/speed.py': '"""Time submission folders row by row on one CPU thread, next to the official baseline.\n'
                      '\n'
                      '    python -m pipeline.speed runs/NAME/submission runs/ens/submission\n'
                      '\n'
                      "The test set is 39.4M rows in 60 min (~91 us/row on the grader's CPU). The baseline is known "
                      'to\n'
                      'fit, so the ratio to the baseline is the number to watch; absolute times depend on this '
                      'machine.\n'
                      '"""\n'
                      'import os\n'
                      '\n'
                      'for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):\n'
                      '    os.environ[_v] = "1"\n'
                      '\n'
                      'import argparse\n'
                      'import importlib.util\n'
                      'import sys\n'
                      'import time\n'
                      '\n'
                      'import numpy as np\n'
                      'import pyarrow.parquet as pq\n'
                      '\n'
                      'from utils import FEATURE_COLUMNS, WARMUP, DataPoint\n'
                      '\n'
                      'TEST_ROWS = 39_400_000\n'
                      '\n'
                      '\n'
                      'def load_model(folder, i):\n'
                      '    folder = os.path.abspath(folder)\n'
                      '    sys.path.insert(0, folder)\n'
                      '    spec = importlib.util.spec_from_file_location(f"solution_{i}", os.path.join(folder, '
                      '"solution.py"))\n'
                      '    mod = importlib.util.module_from_spec(spec)\n'
                      '    spec.loader.exec_module(mod)\n'
                      '    return mod.PredictionModel()\n'
                      '\n'
                      '\n'
                      'def main():\n'
                      '    ap = argparse.ArgumentParser()\n'
                      '    ap.add_argument("folders", nargs="+")\n'
                      '    ap.add_argument("--rows", type=int, default=20000)\n'
                      '    args = ap.parse_args()\n'
                      '\n'
                      '    tbl = pq.ParquetFile("datasets/valid.parquet").read_row_group(0, '
                      'columns=list(FEATURE_COLUMNS))\n'
                      '    X = np.column_stack([tbl[c].to_numpy() for c in '
                      'FEATURE_COLUMNS]).astype(np.float32)[:args.rows]\n'
                      '\n'
                      '    results = []\n'
                      '    for i, folder in enumerate(["baseline"] + args.folders):\n'
                      '        model = load_model(folder, i)\n'
                      '        t = time.perf_counter()\n'
                      '        for step in range(len(X)):\n'
                      '            model.predict(DataPoint(0, step, step >= WARMUP, X[step]))\n'
                      '        us = (time.perf_counter() - t) / len(X) * 1e6\n'
                      '        results.append((folder, us))\n'
                      '        base = results[0][1]\n'
                      '        print(f"{folder:50s} {us:7.1f} us/row  {us / base:5.2f}x baseline  "\n'
                      '              f"~{us * TEST_ROWS / 60e6:5.1f} min for the test set here", flush=True)\n'
                      '\n'
                      '\n'
                      'if __name__ == "__main__":\n'
                      '    main()\n',
 'pipeline/teacher.py': '"""Write a teacher\'s predictions on every training sequence, as the target for '
                        'distillation.\n'
                        '\n'
                        '    python -m pipeline.teacher --ensemble runs/ens/cost99/submission/ensemble.json --out '
                        'teacher.npy\n'
                        '    python -m pipeline.teacher --ckpts runs/sweep2/s1_wmse_h128/best.pt '
                        'runs/sweep2/c3_*/best.pt --out t_in2.npy\n'
                        '    python -m pipeline.teacher --ckpts "runs/sweep4/f*/best.pt" --oof --out t_oof.npy\n'
                        '\n'
                        'Output: float16 .npy of shape [n_train_seqs, 20000, 2] (~850 MB), indexed by train.parquet '
                        'row group,\n'
                        'read by the trainer (Common.teacher / ExpConfig.teacher).\n'
                        '\n'
                        'In-sample (default): every member predicts every sequence, including the ones it trained on.\n'
                        '--oof: members were trained with ExpConfig.fold; sequence i is predicted only by the members '
                        'whose\n'
                        'left-out fold is i % n_folds, so every training sequence gets honest (out-of-fold) '
                        'predictions.\n'
                        'With --ckpts, members are rescaled to a common prediction spread (measured on the first '
                        'batch).\n'
                        '"""\n'
                        'import argparse\n'
                        'import glob\n'
                        'import json\n'
                        'import time\n'
                        '\n'
                        'import numpy as np\n'
                        'import torch\n'
                        '\n'
                        'from .data import SeqStore\n'
                        'from .fast_eval import load_ckpt, net_inputs, pick_device, predict_batch\n'
                        '\n'
                        '\n'
                        'def main():\n'
                        '    ap = argparse.ArgumentParser()\n'
                        '    src = ap.add_mutually_exclusive_group(required=True)\n'
                        '    src.add_argument("--ensemble", help="ensemble.json written by pipeline.ensemble '
                        '--export")\n'
                        '    src.add_argument("--ckpts", nargs="+", help="best.pt paths/globs, averaged")\n'
                        '    ap.add_argument("--oof", action="store_true", help="out-of-fold: use each member only on '
                        'its left-out fold")\n'
                        '    ap.add_argument("--out", required=True)\n'
                        '    ap.add_argument("--data", default="datasets/train.parquet")\n'
                        '    ap.add_argument("--limit", type=int, default=None, help="only predict the first N '
                        'sequences (tests)")\n'
                        '    ap.add_argument("--threads", type=int, default=8)\n'
                        '    ap.add_argument("--device", default="auto")\n'
                        '    args = ap.parse_args()\n'
                        '    device = pick_device(args.device)\n'
                        '\n'
                        '    if args.ensemble:\n'
                        '        spec = json.load(open(args.ensemble))\n'
                        '        paths, scales = spec["members"], np.asarray(spec["scales"])\n'
                        '    else:\n'
                        '        paths = [p for c in args.ckpts for p in sorted(glob.glob(c))]\n'
                        '        scales = None\n'
                        '    cks = [torch.load(p, map_location="cpu", weights_only=False) for p in paths]\n'
                        '    nets = [load_ckpt(p).eval().to(device) for p in paths]\n'
                        '    folds = [ck.get("exp", {}).get("fold") for ck in cks]\n'
                        '    if args.oof:\n'
                        '        assert all(f is not None for f in folds), "--oof needs members trained with a fold"\n'
                        '        n_folds = cks[0].get("n_folds") or max(folds) + 1\n'
                        '    print(f"{len(nets)} members" + (f", folds {sorted(set(folds))}" if args.oof else ""), '
                        'flush=True)\n'
                        '\n'
                        '    store = SeqStore(args.data, args.threads)\n'
                        '    n = store.n_seqs\n'
                        '    out = np.lib.format.open_memmap(args.out, mode="w+", dtype=np.float16, shape=(n, 20000, '
                        '2))\n'
                        '    ids = list(range(args.limit or n))\n'
                        '    if args.limit:\n'
                        '        out[args.limit:] = 0\n'
                        '    t, done = time.time(), 0\n'
                        '    for X, _, _ in store.batches(ids, 128):\n'
                        '        cache = {}  # each member gets its own extra features, computed once per batch and '
                        'kind\n'
                        '        P = np.stack([predict_batch(net, net_inputs(net, X, cache), device) for net in '
                        'nets])  # [K, B, T, 2]\n'
                        '        if scales is None:  # common spread, fixed from the first batch\n'
                        '            sd = P[:, :, 99:].std(axis=(1, 2))  # [K, 2]\n'
                        '            scales = sd.mean(0, keepdims=True) / sd\n'
                        '        P = P * scales[:, None, None, :]\n'
                        '        if args.oof:\n'
                        '            batch_fold = np.arange(done, done + len(X)) % n_folds\n'
                        '            pred = np.zeros(P.shape[1:], np.float32)\n'
                        '            for f in range(n_folds):\n'
                        '                members = [k for k in range(len(nets)) if folds[k] == f]\n'
                        '                rows = batch_fold == f\n'
                        '                pred[rows] = P[members][:, rows].mean(0)\n'
                        '        else:\n'
                        '            pred = P.mean(0)\n'
                        '        out[done:done + len(X)] = pred.astype(np.float16)\n'
                        '        done += len(X)\n'
                        '        if done % 1280 == 0 or done == len(ids):\n'
                        '            print(f"{done}/{len(ids)} sequences, {time.time() - t:.0f}s", flush=True)\n'
                        '    out.flush()\n'
                        '    print(f"teacher ({len(nets)} models{\', out-of-fold\' if args.oof else \'\'}) -> '
                        '{args.out}")\n'
                        '\n'
                        '\n'
                        'if __name__ == "__main__":\n'
                        '    main()\n',
 'pipeline/test_export.py': '"""Check that exported ONNX step graphs + solution.py reproduce PyTorch predictions row '
                            'by row.\n'
                            '\n'
                            '    python -m pipeline.test_export            # random GRU / diff-GRU / LSTM / minGRU / '
                            'TCN / feature kinds / aux head / ensembles\n'
                            '"""\n'
                            'import importlib.util\n'
                            'import os\n'
                            'import shutil\n'
                            'import tempfile\n'
                            '\n'
                            'import numpy as np\n'
                            'import pyarrow.parquet as pq\n'
                            'import torch\n'
                            '\n'
                            'from utils import FEATURE_COLUMNS, WARMUP, DataPoint\n'
                            '\n'
                            'from .fast_eval import predict_batch\n'
                            'from .features import KINDS, N_EXTRA\n'
                            'from .model import EnsembleNet, GRUNet, MinGRUNet, TCNNet, export_onnx\n'
                            '\n'
                            'HERE = os.path.dirname(os.path.abspath(__file__))\n'
                            '\n'
                            '\n'
                            'def run_rows(net, X):\n'
                            '    d = tempfile.mkdtemp()\n'
                            '    export_onnx(net, os.path.join(d, "model.onnx"))\n'
                            '    shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(d, '
                            '"solution.py"))\n'
                            '    shutil.copy(os.path.join(HERE, "features.py"), os.path.join(d, "wnn_features.py"))\n'
                            '    spec = importlib.util.spec_from_file_location("sol_test", os.path.join(d, '
                            '"solution.py"))\n'
                            '    mod = importlib.util.module_from_spec(spec)\n'
                            '    spec.loader.exec_module(mod)\n'
                            '    m = mod.PredictionModel()\n'
                            '    out = np.zeros((len(X), 2), np.float32)\n'
                            '    for seq in (0, 1):  # two sequences: also checks the state reset\n'
                            '        for t in range(len(X)):\n'
                            '            p = m.predict(DataPoint(seq, t, t >= WARMUP, X[t]))\n'
                            '            if p is not None:\n'
                            '                out[t] = p\n'
                            '    shutil.rmtree(d)\n'
                            '    return out\n'
                            '\n'
                            '\n'
                            'def main():\n'
                            '    torch.manual_seed(0)\n'
                            '    tbl = pq.ParquetFile("datasets/valid.parquet").read_row_group(0, '
                            'columns=list(FEATURE_COLUMNS))\n'
                            '    X = np.column_stack([tbl[c].to_numpy() for c in '
                            'FEATURE_COLUMNS]).astype(np.float32)[:3000]\n'
                            '    a, b, c = GRUNet(96), GRUNet(64, use_diff=True), GRUNet(80, layers=3, cell="lstm")\n'
                            '    d, e = GRUNet(64, n_extra=N_EXTRA), GRUNet(48, use_diff=True, n_extra=N_EXTRA)\n'
                            '    # minGRU: 3000 rows cross several scan blocks and chunks; TCN 3**6 = 729-row '
                            'receptive field crosses chunks\n'
                            '    f, g = MinGRUNet(64, layers=3), MinGRUNet(48, layers=2, use_diff=True, '
                            'n_extra=N_EXTRA)\n'
                            '    t1, t2 = TCNNet(32, layers=6, kernel=3), TCNNet(24, layers=8, kernel=2, '
                            'use_diff=True, n_extra=N_EXTRA)\n'
                            '    nets = {"gru": a, "gru_diff": b, "lstm_3layer": c, "gru_feats": d, "gru_diff_feats": '
                            'e,\n'
                            '            "mingru": f, "mingru_diff_feats": g, "tcn_k3": t1, "tcn_k2_diff_feats": t2,\n'
                            '            "ensemble": EnsembleNet([a, b, c], np.array([[1.0, 1.1], [0.9, 1.2], [1.3, '
                            '0.7]])),\n'
                            '            "ensemble_feats": EnsembleNet([a, d, e], np.array([[1.0, 1.0], [0.8, 1.2], '
                            '[1.1, 0.9]])),\n'
                            '            "ensemble_mixed": EnsembleNet([a, f, t1, t2], np.array([[1.0, 1.0], [0.8, '
                            '1.2], [1.1, 0.9], [0.9, 1.0]]))}\n'
                            '    n1, n2 = GRUNet(64, n_extra=KINDS["norm"], feat_kind="norm"), GRUNet(48, '
                            'n_extra=KINDS["act+norm"], feat_kind="act+norm")\n'
                            '    n3 = GRUNet(56, use_diff=True, n_extra=KINDS["norm"], feat_kind="norm", n_aux=7)  # '
                            'aux head: not in the export\n'
                            '    nets.update({"gru_norm": n1, "gru_act+norm": n2, "gru_diff_norm_aux": n3,\n'
                            '                 "ensemble_norm": EnsembleNet([a, n1, n3], np.array([[1.0, 1.0], [0.8, '
                            '1.2], [1.1, 0.9]]))})\n'
                            '    for name, net in nets.items():\n'
                            '        net.eval()\n'
                            '        ref = predict_batch(net, X[None], torch.device("cpu"), chunk=700)[0]\n'
                            '        got = run_rows(net, X)\n'
                            '        err = np.abs(ref[WARMUP:] - got[WARMUP:]).max()\n'
                            '        print(f"{name:18s} max |torch - onnx| = {err:.2e}  {\'OK\' if err < 1e-4 else '
                            '\'MISMATCH\'}")\n'
                            '\n'
                            '\n'
                            'if __name__ == "__main__":\n'
                            '    main()\n',
 'pipeline/train.py': '"""Train one stateful GRU (thin wrapper over trainer.train_many; see multi_train.py for '
                      'several).\n'
                      '\n'
                      '    python -m pipeline.train --name quick --train-seqs 1000 --epochs 2\n'
                      '    python -m pipeline.train --name gru128_wmse --epochs 4\n'
                      '\n'
                      'Outputs in runs/NAME/: best.pt, log.jsonl, results.json, submission/{solution.py, model.onnx}.\n'
                      '"""\n'
                      'import argparse\n'
                      'import dataclasses\n'
                      '\n'
                      'from .multi_train import add_common_flags, add_flags\n'
                      'from .trainer import Common, ExpConfig, train_many\n'
                      '\n'
                      '\n'
                      'def main():\n'
                      '    ap = argparse.ArgumentParser()\n'
                      '    shared = ("seed", "chunk")  # one flag each for a single model: used for the model and the '
                      'data stream\n'
                      '    exp_fields = {f.name for f in dataclasses.fields(ExpConfig)} - set(shared)\n'
                      '    ap.add_argument("--name", required=True)\n'
                      '    add_flags(ap, ExpConfig, skip=("name", *shared))\n'
                      '    add_common_flags(ap)\n'
                      '    args = vars(ap.parse_args())\n'
                      '    exp = ExpConfig(**{k: args.pop(k) for k in list(args) if k in exp_fields}, '
                      'seed=args["seed"])\n'
                      '    train_many([exp], Common(**args))\n'
                      '\n'
                      '\n'
                      'if __name__ == "__main__":\n'
                      '    main()\n',
 'pipeline/trainer.py': '"""Shared training loop: one data stream feeds any number of experiments.\n'
                        '\n'
                        'Every batch of whole sequences is read from parquet once and then used to train each model '
                        'in\n'
                        'turn (truncated BPTT, hidden state carried between `chunk`-step pieces, loss on steps >= '
                        '99).\n'
                        'Evaluation is shared the same way: one pass over the eval data scores all models.\n'
                        '\n'
                        'Optional per model: EMA of the weights (evaluated and saved instead of the raw weights),\n'
                        "distillation towards a teacher's predictions on the training set (see pipeline/teacher.py), "
                        'its own\n'
                        'number of passes, and a data fold to leave out (out-of-fold teachers: sequences with\n'
                        'id % n_folds == fold are skipped by that model), extra input features of a kind '
                        '(pipeline/features.py),\n'
                        'and auxiliary targets (aux_targets) learned by a training-only head.\n'
                        '"""\n'
                        'import copy\n'
                        'import json\n'
                        'import math\n'
                        'import os\n'
                        'import shutil\n'
                        'import time\n'
                        'from dataclasses import asdict, dataclass, field\n'
                        '\n'
                        'import numpy as np\n'
                        'import torch\n'
                        '\n'
                        'from utils import WARMUP\n'
                        '\n'
                        'from .data import SeqStore, split_train\n'
                        'from .features import offline as extra_features, width as feat_width\n'
                        'from .fast_eval import evaluate_many\n'
                        'from .model import BASELINE_MACS, GRUNet, build_net, compute_loss, detach_state, '
                        'distill_loss, export_onnx, load_baseline, macs\n'
                        '\n'
                        'HERE = os.path.dirname(os.path.abspath(__file__))\n'
                        'AUX_CHANGES = [(0, 10), (0, 50), (0, 200), (0, 1000), (1, 50), (1, 200)]  # (instrument, rows '
                        'ahead)\n'
                        "AUX_VOL = 100  # plus: mean |change| of i0's average price over the next AUX_VOL rows\n"
                        'N_AUX = len(AUX_CHANGES) + 1\n'
                        '\n'
                        '\n'
                        'def aux_targets(X):\n'
                        '    """Auxiliary targets from the raw inputs X [B,T,>=112] (torch): future change of each '
                        "instrument's average\n"
                        "    price column (i0: 0-21, i1: 52-73) at several horizons, and i0's near-future activity. -> "
                        'A, V [B,T,N_AUX]:\n'
                        '    targets scaled to unit spread over the batch (clipped to +-5) and where they exist (V, '
                        'rows with enough future)."""\n'
                        '    B, T = X.shape[:2]\n'
                        '    mids = [X[..., 0:22].mean(-1), X[..., 52:74].mean(-1)]\n'
                        '    A, V = X.new_zeros(B, T, N_AUX), torch.zeros(B, T, N_AUX, dtype=torch.bool, '
                        'device=X.device)\n'
                        '    for k, (i, h) in enumerate(AUX_CHANGES):\n'
                        '        A[:, :T - h, k] = mids[i][:, h:] - mids[i][:, :-h]\n'
                        '        V[:, :T - h, k] = True\n'
                        '    step = (mids[0][:, 1:] - mids[0][:, :-1]).abs()  # step[t] = |m[t+1] - m[t]|\n'
                        '    c = torch.cat([step.new_zeros(B, 1), step.cumsum(1)], 1)  # c[t] = sum of step[:t]\n'
                        '    n = T - AUX_VOL\n'
                        '    A[:, :n, -1] = (c[:, AUX_VOL:AUX_VOL + n] - c[:, :n]) / AUX_VOL\n'
                        '    V[:, :n, -1] = True\n'
                        '    for k in range(N_AUX):\n'
                        '        a = A[..., k][V[..., k]]\n'
                        '        A[..., k] = ((A[..., k] - a.mean()) / (a.std() + 1e-8)).clamp(-5, 5) * V[..., k]\n'
                        '    return A, V\n'
                        '\n'
                        '\n'
                        '@dataclass\n'
                        'class ExpConfig:\n'
                        '    """Per-model settings. Everything about the data stream is shared (see Common)."""\n'
                        '    name: str\n'
                        '    hidden: int = 128\n'
                        '    layers: int = 2\n'
                        '    diff: bool = False\n'
                        '    dropout: float = 0.0\n'
                        '    init_baseline: bool = False\n'
                        '    loss: str = "wmse"          # wmse | pearson | mix\n'
                        '    alpha: float = 0.1          # wmse weight in mix\n'
                        '    lr: float = 2e-3\n'
                        '    wd: float = 1e-4\n'
                        '    seed: int = 0\n'
                        '    cell: str = "gru"           # gru | lstm\n'
                        '    chunk: int | None = None    # TBPTT length for this model (default: Common.chunk)\n'
                        '    ema: float = 0.0            # >0: keep an EMA of the weights with this decay; eval/save '
                        'the EMA\n'
                        '    distill: float = 0.0        # weight of the teacher loss (0 = off, 1 = teacher only); '
                        'needs a teacher\n'
                        '    distill_weight: str = "y"   # teacher-loss row weights: y (|clipped target|, like the '
                        'metric) | none | teacher\n'
                        "    teacher: str | None = None  # this model's teacher .npy (default: Common.teacher)\n"
                        '    epochs: int | None = None   # passes for this model (default: Common.epochs)\n'
                        '    fold: int | None = None     # leave out training sequences with id % Common.n_folds == '
                        'fold\n'
                        '    mask_weight: float = 0.0    # lambda: row weight (1 - lambda) + lambda * p^mask_power / '
                        'mean(p^mask_power)\n'
                        '    mask_power: float = 1.0     # p = predicted p(scored) from Common.row_weights '
                        '(pipeline/mask_model.py)\n'
                        '    feats: bool | str = False   # extra causal input features: a kind from features.KINDS '
                        '(act | norm | act+norm);\n'
                        '                                # True = act (the 36 activity / EMA features)\n'
                        '    aux: float = 0.0            # >0: weight of the auxiliary-target loss (aux_targets; GRU '
                        'only, not exported)\n'
                        '    micro_batch: int | None = None  # sequences per optimizer step for this model (default: '
                        'the whole data batch)\n'
                        '    arch: str = "gru"           # gru (cell gru|lstm) | mingru | tcn (see pipeline/model.py)\n'
                        '    kernel: int = 2             # tcn only: kernel size; dilations kernel**l, receptive field '
                        'kernel**layers rows\n'
                        '\n'
                        '\n'
                        '@dataclass\n'
                        'class Common:\n'
                        '    epochs: int = 4\n'
                        '    batch: int = 256            # sequences per batch\n'
                        '    chunk: int = 500            # TBPTT length\n'
                        '    clip_grad: float = 1.0\n'
                        '    train_seqs: int | None = None\n'
                        '    holdout: int = 500\n'
                        '    valid_seqs: int = 400\n'
                        '    evals_per_epoch: int = 2\n'
                        '    seed: int = 0               # data order + holdout split\n'
                        '    device: str = "auto"\n'
                        '    threads: int = 4            # parquet reader threads\n'
                        '    cache_eval: bool = False    # keep eval data in RAM (~9 MB per sequence)\n'
                        '    cache_train: bool = False   # keep training sequences in RAM after first read\n'
                        '    data_dir: str = "datasets"\n'
                        '    out_dir: str = "runs"\n'
                        '    teacher: str | None = None  # teacher predictions on train.parquet (.npy, [n_seqs, 20000, '
                        '2]) for distillation\n'
                        '    n_folds: int = 4            # for ExpConfig.fold\n'
                        '    row_weights: str | None = None  # p(scored) per train row (.npy [n_seqs, 20000]) for '
                        'ExpConfig.mask_weight\n'
                        '    valid_split: str | None = None  # split.json from mask_model: evaluate and select on '
                        'validation half B\n'
                        '    extra: dict = field(default_factory=dict)\n'
                        '\n'
                        '\n'
                        'class Experiment:\n'
                        '    def __init__(self, cfg: ExpConfig, common: Common, device, n_batches):\n'
                        '        self.cfg = cfg\n'
                        '        self.chunk = cfg.chunk or common.chunk\n'
                        '        self.epochs = cfg.epochs or common.epochs\n'
                        '        self.teacher_path = (cfg.teacher or common.teacher) if cfg.distill > 0 else None\n'
                        '        assert cfg.distill == 0 or self.teacher_path, f"{cfg.name}: distill > 0 needs a '
                        'teacher"\n'
                        '        self.micro = cfg.micro_batch or common.batch\n'
                        '        total_steps = (self.epochs * n_batches * math.ceil(common.batch / self.micro)  # '
                        'upper bound on steps\n'
                        '                       * math.ceil(20000 / self.chunk))\n'
                        '        self.kind = "act" if cfg.feats is True else (cfg.feats or None)\n'
                        '        n_extra = feat_width(self.kind)\n'
                        '        assert cfg.aux == 0 or cfg.arch == "gru", "auxiliary targets: GRU only"\n'
                        '        torch.manual_seed(cfg.seed)\n'
                        '        if cfg.init_baseline:\n'
                        '            assert cfg.arch == "gru" and not cfg.diff and cfg.hidden == 128 and cfg.layers == '
                        '2, "baseline shape is fixed"\n'
                        '            self.net = load_baseline("baseline/baseline.onnx")\n'
                        '        elif cfg.arch == "gru":\n'
                        '            self.net = GRUNet(cfg.hidden, cfg.layers, cfg.diff, cfg.dropout, cfg.cell, '
                        'n_extra=n_extra,\n'
                        '                              feat_kind=self.kind, n_aux=N_AUX if cfg.aux > 0 else 0)\n'
                        '        else:\n'
                        '            self.net = build_net(dict(arch=cfg.arch, hidden=cfg.hidden, layers=cfg.layers, '
                        'use_diff=cfg.diff,\n'
                        '                                      dropout=cfg.dropout, n_extra=n_extra, '
                        'feat_kind=self.kind,\n'
                        '                                      **({"kernel": cfg.kernel} if cfg.arch == "tcn" else '
                        '{})))\n'
                        '        self.net.to(device)\n'
                        '        self.ema_net = copy.deepcopy(self.net) if cfg.ema > 0 else None\n'
                        '        self.n_updates = 0\n'
                        '        self.total_steps = total_steps\n'
                        '        self.opt = torch.optim.AdamW(self.net.parameters(), lr=cfg.lr, weight_decay=cfg.wd)\n'
                        '        self.sched = torch.optim.lr_scheduler.OneCycleLR(self.opt, max_lr=cfg.lr, '
                        'total_steps=total_steps,\n'
                        '                                                         pct_start=max(0.05, 3 / '
                        'total_steps))\n'
                        '        self.dir = os.path.join(common.out_dir, cfg.name)\n'
                        '        os.makedirs(self.dir, exist_ok=True)\n'
                        '        self.log_f = open(os.path.join(self.dir, "log.jsonl"), "a")\n'
                        '        self.best = -1.0\n'
                        '        self.params = sum(p.numel() for p in self.net.parameters())\n'
                        '        self.cost = macs(self.net.config) / BASELINE_MACS\n'
                        '\n'
                        '    @property\n'
                        '    def eval_net(self):\n'
                        '        return self.ema_net if self.ema_net is not None else self.net\n'
                        '\n'
                        '    @torch.no_grad()\n'
                        '    def update_ema(self):\n'
                        '        self.n_updates += 1\n'
                        '        if self.ema_net is None:\n'
                        '            return\n'
                        '        decay = min(self.cfg.ema, (1 + self.n_updates) / (10 + self.n_updates))  # warm-up\n'
                        '        for pe, p in zip(self.ema_net.parameters(), self.net.parameters()):\n'
                        '            pe.lerp_(p, 1 - decay)\n'
                        '\n'
                        '    def log(self, **kw):\n'
                        '        line = json.dumps(kw)\n'
                        '        self.log_f.write(line + "\\n")\n'
                        '        self.log_f.flush()\n'
                        '        print(f"[{self.cfg.name}] {line}", flush=True)\n'
                        '\n'
                        '\n'
                        'def _store(common, name):\n'
                        '    return SeqStore(os.path.join(common.data_dir, f"{name}.parquet"), '
                        'n_threads=common.threads)\n'
                        '\n'
                        '\n'
                        'def train_many(configs: list[ExpConfig], common: Common):\n'
                        '    from .fast_eval import pick_device\n'
                        '\n'
                        '    device = pick_device(common.device)\n'
                        '    if device.type == "cuda":\n'
                        '        torch.backends.cudnn.benchmark = True\n'
                        '    rng = np.random.default_rng(common.seed)\n'
                        '    train, valid = _store(common, "train"), _store(common, "valid")\n'
                        '    if common.cache_train:\n'
                        '        train.enable_cache()\n'
                        '    if common.cache_eval:\n'
                        '        valid.enable_cache()\n'
                        '    train_ids, hold_ids = split_train(train.n_seqs, common.holdout, common.seed)\n'
                        '    if common.train_seqs:\n'
                        '        train_ids = rng.choice(train_ids, common.train_seqs, replace=False)\n'
                        '    valid_B = json.load(open(common.valid_split))["B"] if common.valid_split else None\n'
                        '    valid_ids = (valid_B or list(range(valid.n_seqs)))[:common.valid_seqs]\n'
                        '    hold_store = train\n'
                        '    if common.cache_eval and not common.cache_train:  # cache holdout without caching all of '
                        'train\n'
                        '        hold_store = _store(common, "train")\n'
                        '        hold_store.enable_cache()\n'
                        '\n'
                        '    n_batches = math.ceil(len(train_ids) / common.batch)\n'
                        '    exps = [Experiment(c, common, device, n_batches) for c in configs]\n'
                        '    teachers = {}  # path -> memmap, loaded once and shared by the models that use it\n'
                        '    for path in sorted({e.teacher_path for e in exps if e.teacher_path}):\n'
                        '        teachers[path] = np.load(path, mmap_mode="r")\n'
                        '        assert teachers[path].shape[:2] == (train.n_seqs, 20000), f"teacher {path}: shape '
                        '{teachers[path].shape}"\n'
                        '    row_w, p_norm = None, {}\n'
                        '    if any(e.cfg.mask_weight > 0 for e in exps):\n'
                        '        assert common.row_weights, "mask_weight > 0 needs --row-weights (see '
                        'pipeline/mask_model.py)"\n'
                        '        row_w = np.load(common.row_weights, mmap_mode="r")\n'
                        '        assert row_w.shape == (train.n_seqs, 20000), f"row weights shape {row_w.shape}"\n'
                        '        sample = np.asarray(row_w[np.sort(np.asarray(train_ids))[::20], 99:], '
                        'dtype=np.float32)\n'
                        '        for pw in {e.cfg.mask_power for e in exps if e.cfg.mask_weight > 0}:\n'
                        '            p_norm[pw] = float((sample ** pw).mean())  # so the average row weight stays '
                        'about 1\n'
                        '            print(f"row weights: mean p^{pw:g} = {p_norm[pw]:.4f}", flush=True)\n'
                        '    n_epochs = max(e.epochs for e in exps)\n'
                        '    t0 = time.time()\n'
                        '    for e in exps:\n'
                        '        e.log(event="start", config=asdict(e.cfg), common=asdict(common), device=str(device), '
                        'params=e.params,\n'
                        '              cost=round(e.cost, 3), train_seqs=len(train_ids), holdout=len(hold_ids), '
                        'steps=e.total_steps)\n'
                        '    eval_at = set(np.linspace(0, n_batches, common.evals_per_epoch + 1, dtype=int)[1:] - 1)\n'
                        '\n'
                        '    def run_eval(epoch, bi, group):\n'
                        '        nets = [e.eval_net for e in group]\n'
                        '        vs = evaluate_many(nets, valid, valid_ids, device)\n'
                        '        hs = evaluate_many(nets, hold_store, hold_ids, device)\n'
                        '        for e, v, h in zip(group, vs, hs):\n'
                        "            # with a valid split: select on half B's scored rows (the mask model never saw "
                        'them);\n'
                        "            # otherwise don't select on the visible valid mask alone\n"
                        '            sel = v["scored"] if valid_B else (v["scored"] + h["all"]) / 2\n'
                        '            is_best = sel > e.best\n'
                        '            if is_best:\n'
                        '                e.best = sel\n'
                        '                torch.save({"config": e.net.config, "state_dict": e.eval_net.state_dict(), '
                        '"exp": asdict(e.cfg),\n'
                        '                            "n_folds": common.n_folds},\n'
                        '                           os.path.join(e.dir, "best.pt"))\n'
                        '            e.log(event="eval", epoch=epoch, batch=bi, valid=round(v["scored"], 6), '
                        'valid_all=round(v["all"], 6),\n'
                        '                  holdout=round(h["all"], 6), select=round(sel, 6), best=is_best, '
                        'time=round(time.time() - t0, 1))\n'
                        '            e.net.train()\n'
                        '\n'
                        '    for epoch in range(n_epochs):\n'
                        '        active = [e for e in exps if epoch < e.epochs]\n'
                        '        for e in active:\n'
                        '            e.net.train()\n'
                        '        ids = np.asarray(rng.permutation(train_ids))\n'
                        '        for bi, (X, Y, _) in enumerate(train.batches(list(ids), common.batch)):\n'
                        '            tb = time.time()\n'
                        '            batch_ids = ids[bi * common.batch:(bi + 1) * common.batch]\n'
                        '            Xd = torch.from_numpy(X).to(device, non_blocking=True)  # one host->device copy '
                        'per batch\n'
                        '            # extras once per batch and kind, shared by every model of that kind, joined on '
                        'the GPU (the joined\n'
                        '            # 256-seq batch would be another 3+ GB array in host RAM)\n'
                        '            Xk = {None: Xd, **{k: torch.cat([Xd, torch.from_numpy(extra_features(X, '
                        'k)).to(device)], -1)\n'
                        '                               for k in sorted({e.kind for e in active} - {None})}}\n'
                        '            Ad = Vd = None\n'
                        '            if any(e.cfg.aux > 0 for e in active):\n'
                        '                Ad, Vd = aux_targets(Xd)\n'
                        '            Yd = torch.from_numpy(Y).to(device, non_blocking=True)\n'
                        '            order = torch.as_tensor(np.argsort(np.argsort(batch_ids)), device=device)  # '
                        'memmap reads in file order\n'
                        '            Td = {path: torch.from_numpy(np.asarray(t[np.sort(batch_ids)], '
                        'dtype=np.float32)).to(device)[order]\n'
                        '                  for path, t in teachers.items() if any(e.teacher_path == path for e in '
                        'active)}\n'
                        '            Pd = None\n'
                        '            if row_w is not None and any(e.cfg.mask_weight > 0 for e in active):\n'
                        '                Pd = torch.from_numpy(np.asarray(row_w[np.sort(batch_ids)], '
                        'dtype=np.float32)).to(device)[order]\n'
                        '            for e in active:\n'
                        '                Xe, Ye, Te = Xk[e.kind], Yd, Td.get(e.teacher_path)\n'
                        '                Ae, Ve = (Ad, Vd) if e.cfg.aux > 0 else (None, None)\n'
                        '                We = None\n'
                        '                if e.cfg.mask_weight > 0:\n'
                        '                    lam, pw = e.cfg.mask_weight, e.cfg.mask_power\n'
                        '                    We = (1 - lam) + lam * Pd.clamp(min=0) ** pw / p_norm[pw]\n'
                        "                if e.cfg.fold is not None:  # out-of-fold model: drop its fold's sequences "
                        'from the batch\n'
                        '                    rows = torch.as_tensor(np.flatnonzero(batch_ids % common.n_folds != '
                        'e.cfg.fold), device=device)\n'
                        '                    Xe, Ye = Xe[rows], Yd[rows]\n'
                        '                    Ae, Ve = (Ae[rows], Ve[rows]) if Ae is not None else (None, None)\n'
                        '                    Te = Te[rows] if Te is not None else None\n'
                        '                    We = We[rows] if We is not None else None\n'
                        '                losses = []\n'
                        '                for m0 in range(0, len(Xe), e.micro):  # mini-batches of whole sequences, '
                        'each its own TBPTT pass\n'
                        '                    Xm, Ym = Xe[m0:m0 + e.micro], Ye[m0:m0 + e.micro]\n'
                        '                    Tm = Te[m0:m0 + e.micro] if Te is not None else None\n'
                        '                    Wm = We[m0:m0 + e.micro] if We is not None else None\n'
                        '                    Am, Vm = (Ae[m0:m0 + e.micro], Ve[m0:m0 + e.micro]) if Ae is not None '
                        'else (None, None)\n'
                        '                    h = xp = None\n'
                        '                    for s in range(0, X.shape[1], e.chunk):\n'
                        '                        x, y = Xm[:, s:s + e.chunk], Ym[:, s:s + e.chunk]\n'
                        '                        p, h = e.net(x, h, xp)\n'
                        '                        lo = max(0, WARMUP - s)  # skip warm-up rows\n'
                        '                        if lo < x.shape[1]:\n'
                        '                            pf, yf = p[:, lo:].reshape(-1, 2), y[:, lo:].reshape(-1, 2)\n'
                        '                            rw = Wm[:, s:s + e.chunk][:, lo:].reshape(-1) if Wm is not None '
                        'else None\n'
                        '                            lam = e.cfg.distill\n'
                        '                            loss = compute_loss(e.cfg.loss, pf, yf, e.cfg.alpha, rw) if lam < '
                        '1 else 0.0\n'
                        '                            if lam > 0:\n'
                        '                                tf = Tm[:, s:s + e.chunk][:, lo:].reshape(-1, 2)\n'
                        '                                loss = (1 - lam) * loss + lam * distill_loss(pf, tf, yf, '
                        'e.cfg.distill_weight, rw)\n'
                        '                            if Am is not None:  # auxiliary targets on the same rows, from '
                        'the training-only head\n'
                        '                                pa = e.net.aux_head(e.net.last_out[:, lo:])\n'
                        '                                ta, va = Am[:, s:s + e.chunk][:, lo:], Vm[:, s:s + '
                        'e.chunk][:, lo:]\n'
                        '                                loss = loss + e.cfg.aux * ((pa - ta) ** 2 * va).sum() / '
                        'va.sum().clamp(min=1)\n'
                        '                            e.opt.zero_grad(set_to_none=True)\n'
                        '                            loss.backward()\n'
                        '                            torch.nn.utils.clip_grad_norm_(e.net.parameters(), '
                        'common.clip_grad)\n'
                        '                            e.opt.step()\n'
                        '                            e.sched.step()\n'
                        '                            e.update_ema()\n'
                        '                            losses.append(loss.detach())  # no GPU sync per step\n'
                        '                        h = detach_state(h)\n'
                        '                        xp = x[:, -1]\n'
                        '                e.log(event="batch", epoch=epoch, batch=bi, '
                        'loss=round(float(torch.stack(losses).mean()), 5),\n'
                        '                      lr=round(e.sched.get_last_lr()[0], 6), steps=len(losses))\n'
                        '            print(f"batch {bi + 1}/{n_batches} epoch {epoch + 1}/{n_epochs}: {time.time() - '
                        'tb:.1f}s "\n'
                        '                  f"for {len(active)} models, elapsed {(time.time() - t0) / 60:.1f} min", '
                        'flush=True)\n'
                        '            del Xd, Xk, Yd, Td, Pd, Ad, Vd\n'
                        '            if bi in eval_at:\n'
                        '                run_eval(epoch, bi, active)\n'
                        '\n'
                        '    # final: best checkpoints -> full validation + holdout, ONNX submission folders, summary '
                        'table\n'
                        '    for e in exps:\n'
                        '        ck = torch.load(os.path.join(e.dir, "best.pt"), map_location="cpu", '
                        'weights_only=False)\n'
                        '        e.net.load_state_dict(ck["state_dict"])\n'
                        '        e.net.to(device)\n'
                        '    nets = [e.net for e in exps]\n'
                        '    full_valid = evaluate_many(nets, _store(common, "valid"), range(valid.n_seqs), device)  # '
                        'uncached: 17 GB\n'
                        '    full_hold = evaluate_many(nets, hold_store, hold_ids, device)\n'
                        '    full_B = evaluate_many(nets, _store(common, "valid"), valid_B, device) if valid_B else '
                        '[None] * len(exps)\n'
                        '    rows = []\n'
                        '    for e, v, h, b in zip(exps, full_valid, full_hold, full_B):\n'
                        '        sub = os.path.join(e.dir, "submission")\n'
                        '        os.makedirs(sub, exist_ok=True)\n'
                        '        export_onnx(e.net, os.path.join(sub, "model.onnx"))\n'
                        '        shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(sub, '
                        '"solution.py"))\n'
                        '        shutil.copy(os.path.join(HERE, "features.py"), os.path.join(sub, "wnn_features.py"))\n'
                        '        res = {"valid_full": v, "holdout": h, **({"valid_B": b} if b else {})}\n'
                        '        json.dump(res, open(os.path.join(e.dir, "results.json"), "w"), indent=1)\n'
                        '        e.log(event="done", **res)\n'
                        '        rows.append({"name": e.cfg.name, "params": e.params, "cost": round(e.cost, 2), '
                        '"valid_scored": round(v["scored"], 6),\n'
                        '                     "valid_all": round(v["all"], 6), "holdout_all": round(h["all"], 6),\n'
                        '                     "t0": round(v["t0"], 4), "t1": round(v["t1"], 4),\n'
                        '                     **({"valid_B_scored": round(b["scored"], 6), "valid_B_all": '
                        'round(b["all"], 6)} if b else {})})\n'
                        '    rows.sort(key=lambda r: -r["valid_B_scored"] if "valid_B_scored" in r else '
                        '-(r["valid_scored"] + r["holdout_all"]))\n'
                        '    with open(os.path.join(common.out_dir, "summary.jsonl"), "a") as f:\n'
                        '        for r in rows:\n'
                        '            f.write(json.dumps(r) + "\\n")\n'
                        '    print("\\n" + " | ".join(rows[0].keys()))\n'
                        '    for r in rows:\n'
                        '        print(" | ".join(str(v) for v in r.values()))\n'
                        '    return rows\n'}
os.makedirs('/content/wnn/pipeline', exist_ok=True)
for rel, body in FILES.items():
    open(f'/content/wnn/{rel}', 'w').write(body)
sys.path.insert(0, '/content/wnn')
for m in [m for m in sys.modules if m == 'pipeline' or m.startswith('pipeline.')]:
    del sys.modules[m]  # a previous run in this session may have loaded older code
from pipeline.trainer import ExpConfig, N_AUX
from pipeline.features import KINDS
assert 'aux' in ExpConfig.__dataclass_fields__ and 'norm' in KINDS
print(len(FILES), 'pipeline files written; feature kinds', KINDS, '| aux targets', N_AUX, flush=True)
sh('cd /content/wnn && python -m pipeline.fast_eval --baseline --valid-seqs 300')

CACHE = f'{OUT_DIR}/cache'
ROWW, SPLIT = '/content/wnn/mask_train.npy', f'{OUT_DIR}/mask_model/split.json'
sh(f'cp {CACHE}/mask_train.npy {ROWW} && ls -la {ROWW}')
FLAGS = f'--evals-per-epoch 1 --threads 8 --cache-eval --row-weights {ROWW} --valid-split {SPLIT}'
import pandas as pd

def train(experiments, out, epochs):
    """Train what has no results.json in `out` yet (re-runnable)."""
    done = [e['name'] for e in experiments if os.path.exists(f"{out}/{e['name']}/results.json")]
    todo = [e for e in experiments if e['name'] not in done]
    print(f"\n{len(todo)} to train in {out}: {[e['name'] for e in todo]}" + (f"; already done: {done}" if done else ''), flush=True)
    if not todo:
        return
    drive_writable()
    json.dump(todo, open('experiments.json', 'w'), indent=1)
    sh(f"""set -o pipefail; cd /content/wnn && python -m pipeline.multi_train --config experiments.json --epochs {epochs} {FLAGS} --out-dir {out} 2>&1 | grep -v '"event": "batch"'""")

def table(names, dirs):
    rows = [json.loads(l) for d in dirs if os.path.exists(f'{OUT_DIR}/{d}/summary.jsonl') for l in open(f'{OUT_DIR}/{d}/summary.jsonl')]
    df = pd.DataFrame(rows).drop_duplicates('name', keep='last')
    df = df[df.name.isin(names)].sort_values('valid_B_scored', ascending=False)
    print(df[['name', 'cost', 'valid_B_scored', 'valid_B_all', 'valid_scored', 'holdout_all']].to_string(index=False), flush=True)

# ---- Phase A: new teacher members + tests of auxiliary targets and per-sequence normalised inputs (5 passes)
def tm(name, **kw):  # plain mask-weighted model, no teacher
    return {'name': name, 'mask_weight': 0.5, **kw}
PHASE_A = [tm('t9_h128_s0'), tm('t9_h128_s1', seed=1), tm('t9_h128_s2', seed=2), tm('t9_h128_s3', seed=3),
           tm('t9_h192_drop_s0', hidden=192, dropout=0.1), tm('t9_h192_drop_s1', hidden=192, dropout=0.1, seed=1),
           tm('t9_aux03', aux=0.3), tm('t9_aux1', aux=1.0), tm('t9_norm', feats='norm'), tm('t9_norm_aux03', feats='norm', aux=0.3)]
train(PHASE_A, f'{OUT_DIR}/sweep9_teachers', epochs=5)
table([e['name'] for e in PHASE_A] + ['tw_h128_s0', 'tw_h128_s1', 'tw_h192_drop'], ['sweep6_teachers', 'sweep9_teachers'])
print('compare aux / norm with t9_h128_s0 (same recipe, seed 0); sweep 6 members (also 5 passes): tw_*', flush=True)

# ---- Phase B: teachers
def cached_teacher(name, build):
    local, kept = f'/content/wnn/t_{name}.npy', f'{CACHE}/t_{name}.npy'
    if os.path.exists(kept):
        print(f'teacher {name}: cached in Drive, copying', flush=True); sh(f'cp {kept} {local}')
    else:
        build(local); drive_writable(); sh(f'cp {local} {kept}')

POOL = [f'{OUT_DIR}/sweep6_teachers/*/best.pt', f'{OUT_DIR}/sweep5/plain_m50/best.pt', f'{OUT_DIR}/sweep7/*/best.pt',
        f'{OUT_DIR}/sweep8/mingru*/best.pt', f'{OUT_DIR}/sweep8/tcn*/best.pt', f'{OUT_DIR}/sweep9_teachers/*/best.pt']
def build_t9(local):
    pool = ' '.join(f'"{p}"' for p in POOL)
    sh(f'cd /content/wnn && python -m pipeline.ensemble --ckpts {pool} --valid-split {SPLIT} --select-on validA --threads 8 '
       f'--max-members 12 --budgets 99 --export {OUT_DIR}/teacher_9 2>&1 | grep -vE "Warning|_generic_rnn|warnings.warn"')
    sh(f'cd /content/wnn && python -m pipeline.teacher --ensemble {OUT_DIR}/teacher_9/cost99/submission/ensemble.json --out {local}')
cached_teacher('9', build_t9)
print('teacher w for reference: half B 0.6786 (6 models)', flush=True)

STUDENTS = [f'{OUT_DIR}/sweep6_students/sw_h128/best.pt', f'{OUT_DIR}/sweep6_students/sw_h128_s1/best.pt',
            f'{OUT_DIR}/sweep8/sw_feat_h96/best.pt', f'{OUT_DIR}/sweep8/sw_feat_h128/best.pt',
            f'{OUT_DIR}/sweep8/sw_feat_h128_m75/best.pt', f'{OUT_DIR}/sweep8/sw_feat_h128_s1/best.pt']
cached_teacher('ba', lambda local: sh(f'cd /content/wnn && python -m pipeline.teacher --ckpts {" ".join(STUDENTS)} --out {local}'))

# ---- Phase C: students (10 passes)
def st(name, teacher, **kw):
    return {'name': name, 'distill': 1.0, 'teacher': f'/content/wnn/t_{teacher}.npy', 'mask_weight': 0.5, **kw}
PHASE_C = [st('s9_h128', '9'), st('s9_h128_s1', '9', seed=1), st('s9_aux03', '9', aux=0.3), st('s9_norm', '9', feats='norm'),
           st('sba_h128', 'ba')]
SWEEP = f'{OUT_DIR}/sweep9'
train(PHASE_C, SWEEP, epochs=10)
os.makedirs(f'{OUT_DIR}/zips', exist_ok=True)
for e in PHASE_C:  # upload-ready zips
    d = f"{SWEEP}/{e['name']}/submission"
    if os.path.isdir(d): print(shutil.make_archive(f"{OUT_DIR}/zips/sweep9_{e['name']}", 'zip', d), flush=True)
table([e['name'] for e in PHASE_C] + ['sw_h128', 'sw_h128_s1'], ['sweep6_students', 'sweep9'])
print('references on half B: sw_h128_s1 0.6782 (leaderboard 0.65660), the student of teacher w', flush=True)
sh('cd /content/wnn && python -m pipeline.speed ' + ' '.join(f"{SWEEP}/{e['name']}/submission" for e in PHASE_C))  # grader ~ 0.52 x this


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive is writable
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
CPUs: 12
               total        used        free      shared  buff/cache   available
Mem:              83           2          67           0          14          81
train: BROKEN (FileNotFoundError: [Errno 2] Failed to open local file 'datasets/train.parquet'. Detail: [errno 2] No such file or directory)
valid: BROKEN (FileNotFoundError: [Errno 2] Failed to open local file 'datasets/valid.parquet'. Detail: [errno 2] No such file or directory)
valid_mask: BROKEN (FileNotFoundError: [Errno 2] Failed to open local file 'datasets/valid_mask.parquet'. Detail: [errno 2] No such file or directory)
-rw-r--r-- 1 root root 33737976954 Nov  3 21:48 pack.tar.gz
train: 10607 sequences OK
valid: 1873 sequences OK
valid_mask: 1873 sequences OK
data OK
33G	datasets
14 pipeline files written; fe